# PDF Scan v2 - Section-first rebuild

This notebook currently implements **Phase A** and **Phase B** from `PDF_SCAN_PIPELINE_IMPLEMENTATION_PLAN.md`.

- **Phase A**: config surface, run artifacts under `runs/{run_id}/`, structured logging, per-stage metrics, and a PDF manifest.
- **Phase B**: parser bundle creation with `PyMuPDF`, `pypdf`, optional `Docling`, optional `GROBID`, plus raw per-document diagnostics.
- Later phases will add canonical section construction, query planning, retrieval, reranking, and calibration.

**Ground rules for this version**

- Prefer loud, detailed errors over silent failure.
- Each major code cell ends with a compact QC summary.
- Ranking does not happen yet in this notebook version.


## Step 0 - Edit your inputs

Edit the next cell, then run the notebook top-to-bottom.

What the notebook should produce at this point:

- a stable `run_id`
- `runs/<run_id>/config.json`
- `runs/<run_id>/pdf_manifest.json`
- `runs/<run_id>/logs.jsonl`
- `runs/<run_id>/metrics.json`
- placeholder directories for later phases
- per-document parser bundles under `runs/<run_id>/parser/<doc_id>/`


In [ ]:
# -----------------------------
# USER INPUTS (edit this cell)
# -----------------------------

import json
import re
from pathlib import Path

INPUT_MODE = "small_gold"  # "small_gold" or "manual"

# Benchmark mode: pull chapter + PDFs from the populated small-gold suite.
SMALL_GOLD_SUITE_MANIFEST = r"benchmark/small_gold/manifests/suite_manifest.json"
SMALL_GOLD_CHAPTER_INDEX = 0
SMALL_GOLD_DOC_LIMIT = None
SMALL_GOLD_INCLUDE_DOC_IDS = []
SMALL_GOLD_EXCLUDE_DOC_IDS = []

# Manual mode: keep this for later ad hoc runs outside the benchmark suite.
MANUAL_CHAPTER_TITLE = "Technische Grundlagen: Zero Trust Architecture (ZTA) in Unternehmensnetzwerken"

MANUAL_CHAPTER_DESCRIPTION = """
Ziel ist eine prazise, technische Fundierung von Zero Trust Architecture (ZTA) fur Unternehmens-IT (On-Prem, Cloud, Hybrid),
um spater eine konkrete ZTA-Einfuhrung bewerten und planen zu konnen. Dazu gehoren Begriffsdefinition, Kernprinzipien,
Referenzarchitekturen, Telemetrie, kontinuierliche Bewertung, Migration in Legacy-Umgebungen und messbare Bewertungskriterien.
Produktvergleiche, Buyer's Guides und rein allgemeine Kryptographie-Einfuhrungen gehoren nicht in den Scope.
""".strip()

# Manual mode, option A: explicit PDF list
PDF_SOURCES = [
    # {"label": "paper_1", "path": r"C:\\path\\to\\paper.pdf"},
]

# Manual mode, option B: discover PDFs from a directory when PDF_SOURCES is empty
PDF_DIR = r""
PDF_GLOB = "*.pdf"
PDF_RECURSIVE = False
MAX_PDFS = 20

PIPELINE_VERSION = "pdf_scan_v2"
FORCE_REBUILD_PHASE_A = False


def _fmt_int(x) -> str:
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def _truncate(text: str, max_len: int = 120) -> str:
    s = str(text or "")
    return s if len(s) <= max_len else (s[: max_len - 1] + "...")


def print_section(title: str, width: int = 80, char: str = "=") -> None:
    line = char * width
    print(line)
    print(title)
    print(line)


def print_kv(d: dict, key_width: int = 26) -> None:
    for k, v in d.items():
        print(f"{str(k):<{key_width}} {v}")


def _find_pdf_scan_dir_local() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for base in candidates:
        pdf_scan_dir = base / "pdf-scan"
        if pdf_scan_dir.exists() and pdf_scan_dir.is_dir():
            return pdf_scan_dir.resolve()
        if base.name == "pdf-scan" and base.is_dir():
            return base.resolve()
    raise RuntimeError("Could not resolve the pdf-scan directory from the current working directory.")


def _resolve_local_path(raw: str, *, expect_dir: bool) -> Path:
    p = Path(raw).expanduser()
    pdf_scan_dir = _find_pdf_scan_dir_local()
    repo_root = pdf_scan_dir.parent
    candidates = [p]
    if not p.is_absolute():
        candidates.extend([pdf_scan_dir / p, repo_root / p, Path.cwd().resolve() / p])
    seen = set()
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.add(cand)
        if cand.exists() and ((cand.is_dir() and expect_dir) or (cand.is_file() and not expect_dir)):
            return cand
    return candidates[0].resolve()


def _load_json_local(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


BENCHMARK_SUITE_ID = ""
BENCHMARK_CHAPTER_ID = ""
BENCHMARK_SUITE_MANIFEST_RESOLVED = ""
BENCHMARK_SUITE_ROOT = ""

if INPUT_MODE == "small_gold":
    suite_manifest_path = _resolve_local_path(SMALL_GOLD_SUITE_MANIFEST, expect_dir=False)
    if not suite_manifest_path.exists():
        raise FileNotFoundError(f"Small-gold suite manifest not found: {suite_manifest_path}")

    suite_root = suite_manifest_path.parent.parent
    suite = _load_json_local(suite_manifest_path)
    chapter_specs = list(suite.get("chapter_specs") or [])
    if not chapter_specs:
        raise ValueError("The small-gold suite manifest contains no chapter_specs entries.")

    chapter_index = int(SMALL_GOLD_CHAPTER_INDEX)
    if chapter_index < 0 or chapter_index >= len(chapter_specs):
        raise IndexError(f"SMALL_GOLD_CHAPTER_INDEX out of range: {chapter_index}")

    chapter_path = (suite_root / str(chapter_specs[chapter_index])).resolve()
    chapter_spec = _load_json_local(chapter_path)

    include_doc_ids = {str(x).strip() for x in SMALL_GOLD_INCLUDE_DOC_IDS if str(x).strip()}
    exclude_doc_ids = {str(x).strip() for x in SMALL_GOLD_EXCLUDE_DOC_IDS if str(x).strip()}
    loaded_docs = []
    for rel_path in list(suite.get("documents") or []):
        doc_manifest_path = (suite_root / str(rel_path)).resolve()
        doc_manifest = _load_json_local(doc_manifest_path)
        doc_id = str(doc_manifest.get("doc_id") or "").strip()
        if include_doc_ids and doc_id not in include_doc_ids:
            continue
        if doc_id and doc_id in exclude_doc_ids:
            continue

        pdf_path = (suite_root / str(doc_manifest.get("path") or "")).resolve()
        if not pdf_path.exists():
            raise FileNotFoundError(f"Benchmark PDF not found: {pdf_path}")

        loaded_docs.append(
            {
                "label": str(doc_manifest.get("label") or pdf_path.stem).strip() or pdf_path.stem,
                "path": str(pdf_path),
                "doc_id": doc_id,
            }
        )

    if SMALL_GOLD_DOC_LIMIT is not None:
        loaded_docs = loaded_docs[: int(SMALL_GOLD_DOC_LIMIT)]
    if not loaded_docs:
        raise ValueError("Benchmark mode resolved zero PDFs after include/exclude filtering.")

    CHAPTER_TITLE = str(chapter_spec.get("title") or "").strip()
    CHAPTER_DESCRIPTION = str(chapter_spec.get("description") or "").strip()
    PDF_SOURCES = [{"label": row["label"], "path": row["path"]} for row in loaded_docs]
    PDF_DIR = r""
    PDF_GLOB = "*.pdf"
    PDF_RECURSIVE = False
    MAX_PDFS = len(PDF_SOURCES)

    BENCHMARK_SUITE_ID = str(suite.get("suite_id") or "").strip()
    BENCHMARK_CHAPTER_ID = str(chapter_spec.get("chapter_id") or "").strip()
    BENCHMARK_SUITE_MANIFEST_RESOLVED = str(suite_manifest_path)
    BENCHMARK_SUITE_ROOT = str(suite_root)
elif INPUT_MODE == "manual":
    CHAPTER_TITLE = MANUAL_CHAPTER_TITLE
    CHAPTER_DESCRIPTION = MANUAL_CHAPTER_DESCRIPTION
else:
    raise ValueError(f"Unsupported INPUT_MODE: {INPUT_MODE!r}")

if not str(CHAPTER_TITLE or "").strip():
    raise ValueError("CHAPTER_TITLE must not be empty.")
if not str(CHAPTER_DESCRIPTION or "").strip():
    raise ValueError("CHAPTER_DESCRIPTION must not be empty.")

desc_words = len(re.findall(r"\w+", CHAPTER_DESCRIPTION, flags=re.UNICODE))
source_mode = "SMALL_GOLD" if INPUT_MODE == "small_gold" else ("PDF_SOURCES" if PDF_SOURCES else ("PDF_DIR" if str(PDF_DIR or "").strip() else "unset"))

print_section("User Inputs")
print_kv(
    {
        "input_mode": INPUT_MODE,
        "chapter_title": _truncate(CHAPTER_TITLE, 90),
        "chapter_desc_chars": _fmt_int(len(CHAPTER_DESCRIPTION)),
        "chapter_desc_words": _fmt_int(desc_words),
        "pdf_source_mode": source_mode,
        "pipeline_version": PIPELINE_VERSION,
        "force_rebuild_phase_a": FORCE_REBUILD_PHASE_A,
        "benchmark_suite_id": BENCHMARK_SUITE_ID or "<none>",
        "benchmark_chapter_id": BENCHMARK_CHAPTER_ID or "<none>",
    }
)

print_section("User Inputs - PDF Discovery Config")
print_kv(
    {
        "pdf_sources_count": _fmt_int(len(PDF_SOURCES)),
        "pdf_dir": PDF_DIR or "<empty>",
        "pdf_glob": PDF_GLOB,
        "pdf_recursive": PDF_RECURSIVE,
        "max_pdfs": MAX_PDFS,
    }
)

if INPUT_MODE == "small_gold":
    print_section("User Inputs - Small Gold")
    print_kv(
        {
            "suite_manifest": _truncate(BENCHMARK_SUITE_MANIFEST_RESOLVED, 110),
            "suite_root": _truncate(BENCHMARK_SUITE_ROOT, 110),
            "chapter_index": SMALL_GOLD_CHAPTER_INDEX,
            "include_doc_ids": ", ".join(SMALL_GOLD_INCLUDE_DOC_IDS) if SMALL_GOLD_INCLUDE_DOC_IDS else "<all>",
            "exclude_doc_ids": ", ".join(SMALL_GOLD_EXCLUDE_DOC_IDS) if SMALL_GOLD_EXCLUDE_DOC_IDS else "<none>",
        }
    )

---
# Phase A - Config, env loading, run artifacts, and structured logging
---

In [ ]:
# Phase A.0 - Imports, repo resolution, env loading, and helper functions

import json
import logging
import os
import hashlib
import time
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None


def _find_repo_root_and_notebook_dir() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for base in candidates:
        pdf_scan_dir = base / "pdf-scan"
        if pdf_scan_dir.exists() and pdf_scan_dir.is_dir():
            return base, pdf_scan_dir
        if base.name == "pdf-scan":
            return base.parent, base
    raise RuntimeError("Could not resolve repo root / pdf-scan directory from current working directory.")


REPO_ROOT, NOTEBOOK_DIR = _find_repo_root_and_notebook_dir()

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(NOTEBOOK_DIR / ".env", override=False)

OPENAI_API_KEY = (os.getenv("OPENAI_API_KEY") or "").strip()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def fmt_float(x: Any, nd: int = 3) -> str:
    try:
        return f"{float(x):.{int(nd)}f}"
    except Exception:
        return str(x)


def fmt_ms(ms: Any) -> str:
    try:
        val = float(ms)
    except Exception:
        return str(ms)
    if val < 1000:
        return f"{val:.0f}ms"
    return f"{(val / 1000.0):.2f}s"


def print_table(rows: List[Dict[str, Any]], *, columns: List[str], max_rows: int = 50, max_col_width: int = 60) -> None:
    rows = list(rows or [])
    if not rows:
        print("<empty>")
        return

    show = rows[: int(max_rows)]
    cols = list(columns)

    def cell(row: Dict[str, Any], col: str) -> str:
        value = row.get(col, "")
        if value is None:
            value = ""
        text = str(value).replace("\r", " ").replace("\n", " ")
        return text if len(text) <= max_col_width else (text[: max_col_width - 3] + "...")

    widths = {}
    for col in cols:
        widths[col] = min(
            max(len(col), max(len(cell(r, col)) for r in show)),
            max_col_width,
        )

    header = " | ".join(f"{col:<{widths[col]}}" for col in cols)
    sep = "-+-".join("-" * widths[col] for col in cols)
    print(header)
    print(sep)
    for row in show:
        print(" | ".join(f"{cell(row, col):<{widths[col]}}" for col in cols))
    if len(rows) > len(show):
        print(f"... (+{len(rows) - len(show)} more rows)")


def qc_row(check: str, status: str, value: Any, expected: str, why: str, fix: str) -> Dict[str, Any]:
    return {
        "check": str(check),
        "status": str(status),
        "value": str(value),
        "expected": str(expected),
        "why": str(why),
        "fix": str(fix),
    }


def stable_hash(*parts: str, length: int = 24) -> str:
    payload = "\n".join([(p or "").strip().replace("\r\n", "\n") for p in parts])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[: int(length)]


def _json_default(obj: Any):
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")


def write_json(path: Path, obj: Any, retries: int = 6, sleep_sec: float = 0.25) -> None:
    ensure_dir(path.parent)
    payload = json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default) + "\n"
    last_error = None
    for attempt in range(max(1, int(retries))):
        tmp = path.with_suffix(path.suffix + f".{attempt}.tmp")
        try:
            tmp.write_text(payload, encoding="utf-8")
            tmp.replace(path)
            return
        except PermissionError as e:
            last_error = e
            time.sleep(float(sleep_sec) * float(attempt + 1))
        finally:
            try:
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass
    if last_error is not None:
        raise last_error
    raise RuntimeError(f"Failed to write JSON: {path}")


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def append_jsonl(path: Path, obj: Any) -> None:
    ensure_dir(path.parent)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=_json_default) + "\n")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def inspect_pdf(path: Path) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "page_count": None,
        "has_outline": None,
        "inspect_status": "not_attempted",
    }
    if fitz is None:
        out["inspect_status"] = "fitz_unavailable"
        return out
    try:
        with fitz.open(path) as doc:
            out["page_count"] = int(doc.page_count)
            try:
                out["has_outline"] = bool(doc.get_toc())
            except Exception:
                out["has_outline"] = None
        out["inspect_status"] = "ok"
    except Exception as e:
        out["inspect_status"] = f"error:{type(e).__name__}"
    return out


print_section("Phase A.0 - Environment")
print_kv(
    {
        "repo_root": REPO_ROOT,
        "notebook_dir": NOTEBOOK_DIR,
        "openai_api_key_present": bool(OPENAI_API_KEY),
        "pymupdf_available": bool(fitz is not None),
        "utc_now": utc_now_iso(),
    }
)

In [ ]:
# Phase A.1 - Config models, PDF source resolution, run artifacts, and logging helpers

@dataclass
class PdfSource:
    label: str
    path: Path


@dataclass
class RunArtifacts:
    config_json: Path
    pdf_manifest_json: Path
    query_plan_json: Path
    parser_dir: Path
    normalized_dir: Path
    retrieval_dir: Path
    rerank_dir: Path
    final_dir: Path
    logs_jsonl: Path
    run_log: Path
    metrics_json: Path

    @classmethod
    def from_run_dir(cls, run_dir: Path) -> "RunArtifacts":
        return cls(
            config_json=run_dir / "config.json",
            pdf_manifest_json=run_dir / "pdf_manifest.json",
            query_plan_json=run_dir / "query_plan.json",
            parser_dir=run_dir / "parser",
            normalized_dir=run_dir / "normalized",
            retrieval_dir=run_dir / "retrieval",
            rerank_dir=run_dir / "rerank",
            final_dir=run_dir / "final",
            logs_jsonl=run_dir / "logs.jsonl",
            run_log=run_dir / "run.log",
            metrics_json=run_dir / "metrics.json",
        )


@dataclass
class PipelineConfig:
    pipeline_version: str
    input_mode: str
    chapter_title: str
    chapter_spec_text: str
    runs_root: Path
    openai_api_key_present: bool
    force_rebuild_phase_a: bool
    pdf_sources: List[PdfSource]
    pdf_dir_raw: str
    pdf_glob: str
    pdf_recursive: bool
    max_pdfs: int
    benchmark_suite_manifest: str
    benchmark_suite_id: str
    benchmark_chapter_id: str

    def to_snapshot(self) -> Dict[str, Any]:
        return {
            "pipeline_version": self.pipeline_version,
            "input_mode": self.input_mode,
            "chapter_title": self.chapter_title,
            "chapter_spec_text_chars": len(self.chapter_spec_text),
            "runs_root": self.runs_root,
            "openai_api_key_present": self.openai_api_key_present,
            "force_rebuild_phase_a": self.force_rebuild_phase_a,
            "pdf_sources": [{"label": s.label, "path": str(s.path)} for s in self.pdf_sources],
            "pdf_dir_raw": self.pdf_dir_raw,
            "pdf_glob": self.pdf_glob,
            "pdf_recursive": self.pdf_recursive,
            "max_pdfs": self.max_pdfs,
            "benchmark_suite_manifest": self.benchmark_suite_manifest,
            "benchmark_suite_id": self.benchmark_suite_id,
            "benchmark_chapter_id": self.benchmark_chapter_id,
        }


@dataclass
class RunContext:
    repo_root: Path
    notebook_dir: Path
    run_id: str
    run_dir: Path
    artifacts: RunArtifacts

    def create_artifact_skeleton(self, overwrite: bool = False) -> None:
        ensure_dir(self.run_dir)
        ensure_dir(self.artifacts.parser_dir)
        ensure_dir(self.artifacts.normalized_dir)
        ensure_dir(self.artifacts.retrieval_dir)
        ensure_dir(self.artifacts.rerank_dir)
        ensure_dir(self.artifacts.final_dir)

        placeholders: Dict[Path, Any] = {
            self.artifacts.query_plan_json: {"status": "not_run", "phase": "query_planner"},
            self.artifacts.metrics_json: {"run_id": self.run_id, "stages": {}},
        }
        for path, payload in placeholders.items():
            if overwrite or (not path.exists()):
                write_json(path, payload)

        for path in [self.artifacts.logs_jsonl, self.artifacts.run_log]:
            if overwrite or (not path.exists()):
                ensure_dir(path.parent)
                path.write_text("", encoding="utf-8")

        for rel in [
            self.artifacts.normalized_dir / "documents.jsonl",
            self.artifacts.normalized_dir / "sections.jsonl",
            self.artifacts.normalized_dir / "passages.jsonl",
            self.artifacts.retrieval_dir / "fused_candidates.jsonl",
            self.artifacts.rerank_dir / "cross_encoder.jsonl",
            self.artifacts.final_dir / "output.json",
        ]:
            if overwrite or (not rel.exists()):
                ensure_dir(rel.parent)
                if rel.suffix == ".json":
                    write_json(rel, {"status": "not_run"})
                else:
                    rel.write_text("", encoding="utf-8")


def _resolve_existing_path(raw: str, *, expect_dir: bool) -> Path:
    p = Path(raw).expanduser()
    candidates = [p]
    if not p.is_absolute():
        candidates.extend([NOTEBOOK_DIR / p, REPO_ROOT / p, Path.cwd().resolve() / p])
    seen: List[Path] = []
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.append(cand)
        if cand.exists() and ((cand.is_dir() and expect_dir) or (cand.is_file() and not expect_dir)):
            return cand
    return candidates[0].resolve()


def _normalize_pdf_sources(raw_sources: List[Dict[str, Any]]) -> List[PdfSource]:
    out: List[PdfSource] = []
    seen_labels: Dict[str, int] = {}
    for item in raw_sources or []:
        if not isinstance(item, dict):
            continue
        path_raw = str(item.get("path") or "").strip()
        if not path_raw:
            continue
        path = _resolve_existing_path(path_raw, expect_dir=False)
        if not path.exists():
            raise FileNotFoundError(f"PDF not found: {path}")
        label = str(item.get("label") or path.stem).strip() or path.stem
        n = seen_labels.get(label, 0) + 1
        seen_labels[label] = n
        if n > 1:
            label = f"{label} ({n})"
        out.append(PdfSource(label=label, path=path))
    return out


def resolve_pdf_sources() -> List[PdfSource]:
    explicit = _normalize_pdf_sources(PDF_SOURCES)
    if explicit:
        return explicit[: int(MAX_PDFS)]

    pdf_dir = str(PDF_DIR or "").strip()
    if not pdf_dir:
        raise RuntimeError("No PDFs configured. Set PDF_SOURCES or PDF_DIR.")

    root = _resolve_existing_path(pdf_dir, expect_dir=True)
    if not root.exists():
        raise FileNotFoundError(f"PDF_DIR not found: {root}")

    paths = sorted(root.rglob(PDF_GLOB) if bool(PDF_RECURSIVE) else root.glob(PDF_GLOB))
    paths = [p.resolve() for p in paths if p.is_file()]
    if not paths:
        raise FileNotFoundError(f"No PDFs found in {root} with pattern {PDF_GLOB!r}")

    out: List[PdfSource] = []
    seen_labels: Dict[str, int] = {}
    for path in paths[: int(MAX_PDFS)]:
        label = path.stem
        n = seen_labels.get(label, 0) + 1
        seen_labels[label] = n
        if n > 1:
            label = f"{label} ({n})"
        out.append(PdfSource(label=label, path=path))
    return out


def compute_run_id(chapter_title: str, chapter_spec_text: str, pipeline_version: str, manifest_rows: List[Dict[str, Any]]) -> str:
    doc_parts = [f"{row.get('label')}::{row.get('sha256')}" for row in manifest_rows]
    return stable_hash(pipeline_version, chapter_title, chapter_spec_text, "\n".join(doc_parts), length=24)


def load_metrics(run_ctx: RunContext) -> Dict[str, Any]:
    if run_ctx.artifacts.metrics_json.exists():
        try:
            return read_json(run_ctx.artifacts.metrics_json)
        except Exception:
            return {"run_id": run_ctx.run_id, "stages": {}}
    return {"run_id": run_ctx.run_id, "stages": {}}


def save_metrics(run_ctx: RunContext, metrics: Dict[str, Any]) -> None:
    write_json(run_ctx.artifacts.metrics_json, metrics)


def setup_run_logger(run_ctx: RunContext) -> logging.Logger:
    logger_name = f"pdf_scan_v2.{run_ctx.run_id}"
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    logger.handlers = []
    logger.propagate = False

    fh = logging.FileHandler(run_ctx.artifacts.run_log, encoding="utf-8")
    fh.setLevel(logging.INFO)
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    return logger


def log_event(run_ctx: RunContext, *, stage: str, event: str, **payload: Any) -> None:
    append_jsonl(
        run_ctx.artifacts.logs_jsonl,
        {
            "ts_utc": utc_now_iso(),
            "stage": stage,
            "event": event,
            **payload,
        },
    )


@contextmanager
def stage_timer(run_ctx: RunContext, stage: str):
    t0 = time.perf_counter()
    try:
        yield
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 3)
        metrics = load_metrics(run_ctx)
        metrics.setdefault("stages", {}).setdefault(stage, {})["elapsed_ms"] = elapsed_ms
        metrics["stages"][stage]["failed_at_utc"] = utc_now_iso()
        metrics["stages"][stage]["last_error"] = {"type": type(e).__name__, "message": str(e)}
        save_metrics(run_ctx, metrics)
        logger = logging.getLogger(f"pdf_scan_v2.{run_ctx.run_id}")
        if getattr(logger, "handlers", None):
            logger.info("Stage failed | stage=%s | elapsed_ms=%s | error=%s: %s", stage, elapsed_ms, type(e).__name__, str(e))
        log_event(run_ctx, stage=stage, event="stage_failed", elapsed_ms=elapsed_ms, error_type=type(e).__name__, error_message=str(e))
        raise
    else:
        elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 3)
        metrics = load_metrics(run_ctx)
        metrics.setdefault("stages", {}).setdefault(stage, {})["elapsed_ms"] = elapsed_ms
        metrics["stages"][stage]["finished_at_utc"] = utc_now_iso()
        save_metrics(run_ctx, metrics)
        logger = logging.getLogger(f"pdf_scan_v2.{run_ctx.run_id}")
        if getattr(logger, "handlers", None):
            logger.info("Stage finished | stage=%s | elapsed_ms=%s", stage, elapsed_ms)
        log_event(run_ctx, stage=stage, event="stage_finished", elapsed_ms=elapsed_ms)

In [ ]:
# Phase A.2 - Resolve PDFs, create run context, write artifacts, and print QC summary

resolved_sources = resolve_pdf_sources()
if not resolved_sources:
    raise RuntimeError("resolve_pdf_sources() returned no PDFs.")

pdf_manifest_rows: List[Dict[str, Any]] = []
for src in resolved_sources:
    stat = src.path.stat()
    inspect = inspect_pdf(src.path)
    pdf_manifest_rows.append(
        {
            "label": src.label,
            "path": str(src.path),
            "file_name": src.path.name,
            "size_mb": round(float(stat.st_size) / (1024.0 * 1024.0), 3),
            "sha256": sha256_file(src.path),
            "page_count": inspect.get("page_count"),
            "has_outline": inspect.get("has_outline"),
            "inspect_status": inspect.get("inspect_status"),
            "mtime_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).replace(microsecond=0).isoformat(),
        }
    )

run_id = compute_run_id(CHAPTER_TITLE, CHAPTER_DESCRIPTION, PIPELINE_VERSION, pdf_manifest_rows)
runs_root = ensure_dir((NOTEBOOK_DIR / "runs").resolve())
run_dir = runs_root / run_id
artifacts = RunArtifacts.from_run_dir(run_dir)
run_ctx = RunContext(repo_root=REPO_ROOT, notebook_dir=NOTEBOOK_DIR, run_id=run_id, run_dir=run_dir, artifacts=artifacts)

cfg = PipelineConfig(
    pipeline_version=PIPELINE_VERSION,
    input_mode=INPUT_MODE,
    chapter_title=CHAPTER_TITLE,
    chapter_spec_text=CHAPTER_DESCRIPTION,
    runs_root=runs_root,
    openai_api_key_present=bool(OPENAI_API_KEY),
    force_rebuild_phase_a=bool(FORCE_REBUILD_PHASE_A),
    pdf_sources=resolved_sources,
    pdf_dir_raw=str(PDF_DIR or ""),
    pdf_glob=str(PDF_GLOB or "*.pdf"),
    pdf_recursive=bool(PDF_RECURSIVE),
    max_pdfs=int(MAX_PDFS),
    benchmark_suite_manifest=str(BENCHMARK_SUITE_MANIFEST_RESOLVED or ""),
    benchmark_suite_id=str(BENCHMARK_SUITE_ID or ""),
    benchmark_chapter_id=str(BENCHMARK_CHAPTER_ID or ""),
)

with stage_timer(run_ctx, "phase_a"):
    run_ctx.create_artifact_skeleton(overwrite=bool(FORCE_REBUILD_PHASE_A))
    logger = setup_run_logger(run_ctx)
    logger.info("Phase A initialized | run_id=%s | run_dir=%s", run_ctx.run_id, run_ctx.run_dir)

    write_json(run_ctx.artifacts.config_json, cfg.to_snapshot())
    write_json(
        run_ctx.artifacts.pdf_manifest_json,
        {
            "generated_at_utc": utc_now_iso(),
            "run_id": run_ctx.run_id,
            "pdf_count": len(pdf_manifest_rows),
            "pdfs": pdf_manifest_rows,
        },
    )

    metrics = load_metrics(run_ctx)
    metrics.setdefault("stages", {}).setdefault("phase_a", {}).update(
        {
            "initialized_at_utc": utc_now_iso(),
            "input_mode": INPUT_MODE,
            "pdf_count": len(pdf_manifest_rows),
            "has_openai_api_key": bool(OPENAI_API_KEY),
            "pymupdf_available": bool(fitz is not None),
            "benchmark_suite_id": BENCHMARK_SUITE_ID or "",
            "benchmark_chapter_id": BENCHMARK_CHAPTER_ID or "",
        }
    )
    save_metrics(run_ctx, metrics)

    log_event(
        run_ctx,
        stage="phase_a",
        event="run_initialized",
        run_id=run_ctx.run_id,
        run_dir=str(run_ctx.run_dir),
        input_mode=INPUT_MODE,
        pdf_count=len(pdf_manifest_rows),
        has_openai_api_key=bool(OPENAI_API_KEY),
        benchmark_suite_id=BENCHMARK_SUITE_ID or "",
        benchmark_chapter_id=BENCHMARK_CHAPTER_ID or "",
    )

expected_paths = [
    run_ctx.artifacts.config_json,
    run_ctx.artifacts.pdf_manifest_json,
    run_ctx.artifacts.query_plan_json,
    run_ctx.artifacts.parser_dir,
    run_ctx.artifacts.normalized_dir,
    run_ctx.artifacts.retrieval_dir,
    run_ctx.artifacts.rerank_dir,
    run_ctx.artifacts.final_dir,
    run_ctx.artifacts.logs_jsonl,
    run_ctx.artifacts.run_log,
    run_ctx.artifacts.metrics_json,
]
missing_paths = [str(p) for p in expected_paths if not p.exists()]

artifact_rows = [
    {"artifact": "config_json", "path": run_ctx.artifacts.config_json, "exists": run_ctx.artifacts.config_json.exists()},
    {"artifact": "pdf_manifest_json", "path": run_ctx.artifacts.pdf_manifest_json, "exists": run_ctx.artifacts.pdf_manifest_json.exists()},
    {"artifact": "query_plan_json", "path": run_ctx.artifacts.query_plan_json, "exists": run_ctx.artifacts.query_plan_json.exists()},
    {"artifact": "parser_dir", "path": run_ctx.artifacts.parser_dir, "exists": run_ctx.artifacts.parser_dir.exists()},
    {"artifact": "normalized_dir", "path": run_ctx.artifacts.normalized_dir, "exists": run_ctx.artifacts.normalized_dir.exists()},
    {"artifact": "retrieval_dir", "path": run_ctx.artifacts.retrieval_dir, "exists": run_ctx.artifacts.retrieval_dir.exists()},
    {"artifact": "rerank_dir", "path": run_ctx.artifacts.rerank_dir, "exists": run_ctx.artifacts.rerank_dir.exists()},
    {"artifact": "final_dir", "path": run_ctx.artifacts.final_dir, "exists": run_ctx.artifacts.final_dir.exists()},
    {"artifact": "logs_jsonl", "path": run_ctx.artifacts.logs_jsonl, "exists": run_ctx.artifacts.logs_jsonl.exists()},
    {"artifact": "metrics_json", "path": run_ctx.artifacts.metrics_json, "exists": run_ctx.artifacts.metrics_json.exists()},
]

qc_rows = []
qc_rows.append(
    qc_row(
        check="artifact_skeleton",
        status="OK" if not missing_paths else "FAIL",
        value="all present" if not missing_paths else ("missing: " + ", ".join(missing_paths[:4])),
        expected="all expected artifact paths exist",
        why="later phases rely on deterministic artifact locations",
        fix="re-run Phase A or inspect permissions / path resolution",
    )
)
qc_rows.append(
    qc_row(
        check="openai_api_key",
        status="OK" if bool(OPENAI_API_KEY) else "WARN",
        value=bool(OPENAI_API_KEY),
        expected="True before query-planning and LLM phases",
        why="later phases use the OpenAI API",
        fix="set OPENAI_API_KEY in .env before Phase D",
    )
)
qc_rows.append(
    qc_row(
        check="pymupdf_available",
        status="OK" if bool(fitz is not None) else "WARN",
        value=bool(fitz is not None),
        expected="True",
        why="PyMuPDF is used for PDF inspection and later fallback parsing",
        fix="install PyMuPDF before Phase B if this is False",
    )
)
qc_rows.append(
    qc_row(
        check="pdf_count",
        status="OK" if len(pdf_manifest_rows) >= 1 else "FAIL",
        value=len(pdf_manifest_rows),
        expected=">= 1",
        why="the pipeline needs at least one PDF input",
        fix="add PDFs to PDF_SOURCES or PDF_DIR",
    )
)

RUN_LOGGER = setup_run_logger(run_ctx)
RUN_CONTEXT = run_ctx
CONFIG = cfg
PDF_MANIFEST = pdf_manifest_rows

print_section("Phase A.2 - Run Context")
print_kv(
    {
        "run_id": run_ctx.run_id,
        "run_dir": run_ctx.run_dir,
        "input_mode": INPUT_MODE,
        "pdf_count": len(pdf_manifest_rows),
        "chapter_title": _truncate(CHAPTER_TITLE, 90),
        "pipeline_version": PIPELINE_VERSION,
        "benchmark_suite_id": BENCHMARK_SUITE_ID or "<none>",
        "benchmark_chapter_id": BENCHMARK_CHAPTER_ID or "<none>",
    }
)

print_section("Phase A.2 - PDF Manifest Preview")
print_table(
    pdf_manifest_rows,
    columns=["label", "file_name", "page_count", "has_outline", "size_mb", "inspect_status"],
    max_rows=20,
)

print_section("Phase A.2 - Artifact Preview")
print_table(artifact_rows, columns=["artifact", "exists", "path"], max_rows=20, max_col_width=70)

print_section("Phase A.2 - QC")
print_table(qc_rows, columns=["check", "status", "value", "expected", "why", "fix"], max_rows=20, max_col_width=46)

---
# Phase B - Parser bundle, raw artifacts, and readability diagnostics
---


In [ ]:
# Phase B.0 - Parser bundle helpers and artifact writers

import importlib.metadata as importlib_metadata
import io
import json
import math
import re
import site
import sys
import time
import traceback
import warnings
from contextlib import redirect_stderr, redirect_stdout
from dataclasses import dataclass
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

OPTIONAL_IMPORT_ERRORS = {}

try:
    import fitz  # PyMuPDF
except Exception as e:
    fitz = None
    OPTIONAL_IMPORT_ERRORS["fitz"] = f"{type(e).__name__}: {e}"

try:
    from pypdf import PdfReader
except Exception as e:
    PdfReader = None
    OPTIONAL_IMPORT_ERRORS["pypdf"] = f"{type(e).__name__}: {e}"

try:
    from docling.document_converter import DocumentConverter
except Exception as e:
    DocumentConverter = None
    OPTIONAL_IMPORT_ERRORS["docling"] = f"{type(e).__name__}: {e}"

try:
    import requests
except Exception as e:
    requests = None
    OPTIONAL_IMPORT_ERRORS["requests"] = f"{type(e).__name__}: {e}"

try:
    from bs4 import BeautifulSoup
except Exception as e:
    BeautifulSoup = None
    OPTIONAL_IMPORT_ERRORS["bs4"] = f"{type(e).__name__}: {e}"


@dataclass
class PhaseBOptions:
    force_rebuild: bool = False
    doc_limit: Optional[int] = None
    include_doc_ids: Optional[List[str]] = None
    exclude_doc_ids: Optional[List[str]] = None
    min_page_words: int = 20
    min_doc_chars: int = 200
    try_docling: bool = True
    docling_page_limit: int = 200
    try_grobid: bool = True
    grobid_page_limit: int = 200
    grobid_base_url: str = ""
    grobid_process_path: str = "/api/processFulltextDocument"
    grobid_timeout_sec: int = 120
    grobid_consolidate_header: int = 0
    grobid_consolidate_citations: int = 0
    grobid_include_raw_citations: int = 0

    def normalized(self) -> "PhaseBOptions":
        return PhaseBOptions(
            force_rebuild=bool(self.force_rebuild),
            doc_limit=None if self.doc_limit is None else int(self.doc_limit),
            include_doc_ids=[str(x).strip() for x in (self.include_doc_ids or []) if str(x).strip()],
            exclude_doc_ids=[str(x).strip() for x in (self.exclude_doc_ids or []) if str(x).strip()],
            min_page_words=int(self.min_page_words),
            min_doc_chars=int(self.min_doc_chars),
            try_docling=bool(self.try_docling),
            docling_page_limit=int(self.docling_page_limit),
            try_grobid=bool(self.try_grobid),
            grobid_page_limit=int(self.grobid_page_limit),
            grobid_base_url=str(self.grobid_base_url or "").strip(),
            grobid_process_path=str(self.grobid_process_path or "/api/processFulltextDocument").strip() or "/api/processFulltextDocument",
            grobid_timeout_sec=int(self.grobid_timeout_sec),
            grobid_consolidate_header=int(self.grobid_consolidate_header),
            grobid_consolidate_citations=int(self.grobid_consolidate_citations),
            grobid_include_raw_citations=int(self.grobid_include_raw_citations),
        )


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def pkg_version(name: str) -> Optional[str]:
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None


def json_safe(obj: Any) -> Any:
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, Enum):
        return obj.value
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [json_safe(v) for v in obj]
    return obj


def write_json_atomic(path: Path, obj: Any, retries: int = 6, sleep_sec: float = 0.25) -> None:
    ensure_dir(path.parent)
    payload = json.dumps(json_safe(obj), ensure_ascii=False, indent=2) + "\n"
    last_error = None
    for attempt in range(max(1, int(retries))):
        tmp = path.with_suffix(path.suffix + f".{attempt}.tmp")
        try:
            tmp.write_text(payload, encoding="utf-8")
            tmp.replace(path)
            return
        except PermissionError as e:
            last_error = e
            time.sleep(float(sleep_sec) * float(attempt + 1))
        finally:
            try:
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass
    if last_error is not None:
        raise last_error
    raise RuntimeError(f"Failed to write JSON atomically: {path}")


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def write_jsonl_rows(path: Path, rows: List[Dict[str, Any]]) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(json_safe(row), ensure_ascii=False) + "\n")


def clean_text(text: Any) -> str:
    s = str(text or "")
    s = s.replace("\xad", "")
    s = s.replace("\u00a0", " ")
    s = s.replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def count_words(text: Any) -> int:
    return len(re.findall(r"\w+", str(text or ""), flags=re.UNICODE))


def slugify(text: str, max_len: int = 64) -> str:
    s = re.sub(r"[^A-Za-z0-9]+", "_", str(text or "").strip().lower()).strip("_")
    s = re.sub(r"_+", "_", s)
    return (s or "doc")[: int(max_len)]


def rel_to_run(run_dir: Path, path: Path) -> str:
    try:
        return str(path.relative_to(run_dir))
    except Exception:
        return str(path)


def short_blob(text: str, max_len: int = 12000) -> str:
    s = str(text or "")
    return s if len(s) <= max_len else (s[: max_len - 1] + "...")


def runtime_env_snapshot() -> Dict[str, Any]:
    try:
        user_site = site.getusersitepackages()
    except Exception:
        user_site = None
    try:
        site_packages = [str(p) for p in site.getsitepackages()]
    except Exception:
        site_packages = []
    return {
        "python_executable": sys.executable,
        "python_version": sys.version.split()[0],
        "python_prefix": sys.prefix,
        "python_base_prefix": getattr(sys, "base_prefix", sys.prefix),
        "cwd": str(Path.cwd()),
        "user_site": user_site,
        "site_packages": site_packages,
        "sys_path_preview": [str(x) for x in sys.path[:15]],
    }


def capture_python_noise(fn: Callable[[], Any]) -> Dict[str, Any]:
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    with redirect_stdout(stdout_buf), redirect_stderr(stderr_buf), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = fn()
    return {
        "result": result,
        "stdout": short_blob(stdout_buf.getvalue()),
        "stderr": short_blob(stderr_buf.getvalue()),
        "warnings": [short_blob(str(w.message), max_len=2000) for w in caught],
    }


def ping_grobid(base_url: str) -> Dict[str, Any]:
    out = {
        "configured": bool(base_url),
        "reachable": False,
        "status": "not_configured",
        "url": base_url,
        "error": None,
    }
    if not base_url:
        return out
    if requests is None:
        out["status"] = "requests_unavailable"
        return out
    try:
        resp = requests.get(base_url.rstrip("/") + "/api/isalive", timeout=10)
        out["reachable"] = bool(resp.ok)
        out["status"] = "alive" if resp.ok else f"http_{resp.status_code}"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


def detect_capabilities(grobid_base_url: str) -> Dict[str, Any]:
    return {
        "generated_at_utc": utc_now_iso(),
        "runtime": runtime_env_snapshot(),
        "fitz_available": bool(fitz is not None),
        "fitz_version": pkg_version("PyMuPDF"),
        "pypdf_available": bool(PdfReader is not None),
        "pypdf_version": pkg_version("pypdf"),
        "docling_available": bool(DocumentConverter is not None),
        "docling_version": pkg_version("docling"),
        "requests_available": bool(requests is not None),
        "requests_version": pkg_version("requests"),
        "bs4_available": bool(BeautifulSoup is not None),
        "bs4_version": pkg_version("beautifulsoup4"),
        "optional_import_errors": dict(OPTIONAL_IMPORT_ERRORS),
        "grobid": ping_grobid(grobid_base_url),
    }


def required_phase_b_kernel_packages(options: PhaseBOptions) -> List[str]:
    packages = ["PyMuPDF", "pypdf"]
    if bool(options.try_docling):
        packages.append("docling")
    if bool(options.try_grobid):
        packages.extend(["requests", "beautifulsoup4"])
    return packages


def missing_phase_b_kernel_packages(capabilities: Dict[str, Any], options: PhaseBOptions) -> List[str]:
    missing: List[str] = []
    if not bool(capabilities.get("fitz_available")):
        missing.append("PyMuPDF")
    if not bool(capabilities.get("pypdf_available")):
        missing.append("pypdf")
    if bool(options.try_docling) and not bool(capabilities.get("docling_available")):
        missing.append("docling")
    if bool(options.try_grobid) and not bool(capabilities.get("requests_available")):
        missing.append("requests")
    if bool(options.try_grobid) and not bool(capabilities.get("bs4_available")):
        missing.append("beautifulsoup4")
    return missing


def compute_doc_id(manifest_row: Dict[str, Any], stable_hash_fn: Optional[Callable[..., str]] = None) -> str:
    stem = slugify(Path(manifest_row.get("file_name") or "document.pdf").stem, max_len=48)
    digest = str(manifest_row.get("sha256") or "")[:12]
    if not digest and stable_hash_fn is not None:
        digest = stable_hash_fn(stem, str(manifest_row.get("path") or ""), length=12)
    digest = digest or "docbundle0000"
    return f"{stem}-{digest}"


def flatten_pypdf_outline(reader: Any, outline: Any) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    def walk(nodes: Any, level: int) -> None:
        for node in nodes or []:
            if isinstance(node, list):
                walk(node, level + 1)
                continue
            title = clean_text(getattr(node, "title", None) or str(node))
            page_num = None
            try:
                page_num = int(reader.get_destination_page_number(node)) + 1
            except Exception:
                page_num = None
            rows.append({"level": int(level), "title": title, "page": page_num})

    if isinstance(outline, list):
        walk(outline, 1)
    return rows


def extract_pypdf_bundle(path: Path) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "available": bool(PdfReader is not None),
        "status": "unavailable",
        "page_count": None,
        "metadata": {},
        "outline": [],
        "error": None,
    }
    if PdfReader is None:
        return out
    try:
        reader = PdfReader(str(path))
        out["page_count"] = int(len(reader.pages))
        out["metadata"] = {str(k): str(v) for k, v in dict(reader.metadata or {}).items()}
        out["outline"] = flatten_pypdf_outline(reader, getattr(reader, "outline", []))
        out["status"] = "ok"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


def normalize_fitz_block(block: Any, page_num: int, block_idx: int) -> Dict[str, Any]:
    row: Dict[str, Any] = {
        "page": int(page_num),
        "block_index": int(block_idx),
        "x0": None,
        "y0": None,
        "x1": None,
        "y1": None,
        "text": "",
        "block_no": None,
        "block_type": None,
        "char_len": 0,
        "word_count": 0,
    }
    if isinstance(block, (list, tuple)):
        vals = list(block)
        if len(vals) >= 4:
            row["x0"], row["y0"], row["x1"], row["y1"] = [float(v) if v is not None else None for v in vals[:4]]
        if len(vals) >= 5:
            row["text"] = clean_text(vals[4])
        if len(vals) >= 6:
            row["block_no"] = vals[5]
        if len(vals) >= 7:
            row["block_type"] = vals[6]
    row["char_len"] = len(row["text"])
    row["word_count"] = count_words(row["text"])
    return row


def extract_fitz_bundle(path: Path, min_page_words: int) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "available": bool(fitz is not None),
        "status": "unavailable",
        "page_count": None,
        "metadata": {},
        "outline": [],
        "pages": [],
        "blocks": [],
        "error": None,
    }
    if fitz is None:
        return out
    try:
        with fitz.open(path) as doc:
            out["page_count"] = int(doc.page_count)
            out["metadata"] = {str(k): str(v) for k, v in dict(doc.metadata or {}).items()}
            try:
                toc = doc.get_toc(simple=True) or []
            except Exception:
                toc = []
            out["outline"] = [
                {
                    "level": int(item[0]),
                    "title": clean_text(item[1]),
                    "page": int(item[2]) if len(item) > 2 and item[2] is not None else None,
                }
                for item in toc
            ]
            for page_index in range(doc.page_count):
                page = doc[page_index]
                try:
                    page_text = clean_text(page.get_text("text", sort=True))
                except TypeError:
                    page_text = clean_text(page.get_text("text"))
                page_word_count = count_words(page_text)
                try:
                    raw_blocks = page.get_text("blocks", sort=True)
                except TypeError:
                    raw_blocks = page.get_text("blocks")

                block_rows = []
                for block_idx, block in enumerate(raw_blocks or []):
                    row = normalize_fitz_block(block, page_index + 1, block_idx)
                    if row["text"]:
                        block_rows.append(row)
                out["blocks"].extend(block_rows)
                out["pages"].append(
                    {
                        "page": int(page_index + 1),
                        "text": page_text,
                        "char_len": len(page_text),
                        "word_count": page_word_count,
                        "has_text": bool(page_word_count > 0),
                        "has_substantive_text": bool(page_word_count >= int(min_page_words)),
                    }
                )
        out["status"] = "ok"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


_DOCLING_CONVERTER = None


def get_docling_converter() -> Any:
    global _DOCLING_CONVERTER
    if DocumentConverter is None:
        return None
    if _DOCLING_CONVERTER is None:
        _DOCLING_CONVERTER = DocumentConverter()
    return _DOCLING_CONVERTER


def extract_docling_bundle(path: Path, page_count: Optional[int], options: PhaseBOptions) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "available": bool(DocumentConverter is not None),
        "enabled": False,
        "status": "unavailable",
        "error": None,
        "stdout": "",
        "stderr": "",
        "warnings": [],
        "result": None,
        "document": None,
        "markdown_preview": None,
    }
    if DocumentConverter is None:
        return out
    if not bool(options.try_docling):
        out["status"] = "disabled"
        return out
    if page_count and int(page_count) > int(options.docling_page_limit):
        out["status"] = "skipped_page_limit"
        out["error"] = f"page_count={page_count} exceeds docling_page_limit={options.docling_page_limit}"
        return out

    out["enabled"] = True
    try:
        captured = capture_python_noise(
            lambda: get_docling_converter().convert(
                path,
                raises_on_error=False,
                max_num_pages=int(options.docling_page_limit),
            )
        )
        res = captured["result"]
        out["stdout"] = captured["stdout"]
        out["stderr"] = captured["stderr"]
        out["warnings"] = captured["warnings"]
        raw_dump = json_safe(res.model_dump()) if hasattr(res, "model_dump") else None
        out["result"] = {
            "status": raw_dump.get("status") if isinstance(raw_dump, dict) else None,
            "errors": raw_dump.get("errors") if isinstance(raw_dump, dict) else None,
            "input": raw_dump.get("input") if isinstance(raw_dump, dict) else None,
            "timings": raw_dump.get("timings") if isinstance(raw_dump, dict) else None,
            "confidence": raw_dump.get("confidence") if isinstance(raw_dump, dict) else None,
        }
        out["status"] = str(out["result"].get("status") or "unknown")
        if out["status"] == "success":
            doc = getattr(res, "document", None)
            if doc is not None:
                out["document"] = json_safe(doc.export_to_dict())
                try:
                    out["markdown_preview"] = short_blob(doc.export_to_markdown(), max_len=8000)
                except Exception:
                    out["markdown_preview"] = None
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
        out["stderr"] = short_blob(traceback.format_exc())
    return out


def should_try_grobid(
    manifest_row: Dict[str, Any],
    page_count: Optional[int],
    options: PhaseBOptions,
    capabilities: Dict[str, Any],
) -> tuple[bool, str]:
    if not bool(options.try_grobid):
        return False, "disabled"
    grobid = capabilities.get("grobid", {})
    if not bool(grobid.get("configured")):
        return False, "not_configured"
    if not bool(grobid.get("reachable")):
        return False, f"service_{grobid.get('status') or 'unreachable'}"
    if page_count and int(page_count) > int(options.grobid_page_limit):
        return False, "skipped_page_limit"
    return True, "pdf_under_page_limit"


def summarize_grobid_xml(xml_text: str) -> Dict[str, Any]:
    xml_text = str(xml_text or "")
    if not xml_text:
        return {"status": "empty"}
    if BeautifulSoup is None:
        return {"status": "bs4_unavailable"}
    try:
        soup = BeautifulSoup(xml_text, "xml")
        head_texts = []
        for tag in soup.find_all("head"):
            txt = clean_text(tag.get_text(" ", strip=True))
            if txt:
                head_texts.append(txt)
        title_tag = soup.find("title")
        return {
            "status": "ok",
            "title": clean_text(title_tag.get_text(" ", strip=True)) if title_tag else None,
            "section_head_count": len(head_texts),
            "section_head_preview": head_texts[:15],
            "has_abstract": bool(soup.find("abstract")),
            "has_bibliography": bool(soup.find("listBibl")),
        }
    except Exception as e:
        return {"status": f"error:{type(e).__name__}", "error": str(e)}


def extract_grobid_bundle(
    path: Path,
    manifest_row: Dict[str, Any],
    page_count: Optional[int],
    options: PhaseBOptions,
    capabilities: Dict[str, Any],
) -> Dict[str, Any]:
    enabled, reason = should_try_grobid(manifest_row, page_count, options, capabilities)
    out: Dict[str, Any] = {
        "available": bool(requests is not None),
        "enabled": bool(enabled),
        "status": "not_attempted",
        "reason": reason,
        "error": None,
        "xml_text": None,
        "summary": None,
    }
    if not enabled:
        out["status"] = reason
        return out
    try:
        with path.open("rb") as f:
            response = requests.post(
                options.grobid_base_url.rstrip("/") + options.grobid_process_path,
                files={"input": (path.name, f, "application/pdf")},
                data={
                    "consolidateHeader": str(int(options.grobid_consolidate_header)),
                    "consolidateCitations": str(int(options.grobid_consolidate_citations)),
                    "includeRawCitations": str(int(options.grobid_include_raw_citations)),
                },
                timeout=int(options.grobid_timeout_sec),
            )
        if not response.ok:
            out["status"] = f"http_{response.status_code}"
            out["error"] = short_blob(response.text, max_len=4000)
            return out
        xml_text = response.text or ""
        out["xml_text"] = xml_text
        out["summary"] = summarize_grobid_xml(xml_text)
        out["status"] = "ok" if xml_text.strip() else "empty_response"
    except Exception as e:
        out["status"] = f"error:{type(e).__name__}"
        out["error"] = str(e)
    return out


def _docling_success_like(status: Any) -> bool:
    return str(status or "") in {"success", "partial_success"}


def compute_phase_b_counts(
    summary_rows: List[Dict[str, Any]],
    capabilities: Dict[str, Any],
    selected_count: int,
) -> Dict[str, Any]:
    low_coverage_docs = [
        row["doc_id"]
        for row in summary_rows
        if row.get("pages_with_text_pct") is None or float(row.get("pages_with_text_pct") or 0.0) < 50.0
    ]
    unreadable_docs = [row["doc_id"] for row in summary_rows if not row.get("readable_without_ocr")]
    cached_docs = [row["doc_id"] for row in summary_rows if row.get("cached")]
    runtime_capability_mismatch_docs = [row["doc_id"] for row in summary_rows if row.get("runtime_capability_mismatch")]
    fallback_docs = [row["doc_id"] for row in summary_rows if row.get("fallback_activated")]
    docling_success_like_docs = [row["doc_id"] for row in summary_rows if _docling_success_like(row.get("docling_status"))]
    docling_success_docs = [row["doc_id"] for row in summary_rows if str(row.get("docling_status") or "") == "success"]
    docling_partial_docs = [row["doc_id"] for row in summary_rows if str(row.get("docling_status") or "") == "partial_success"]
    grobid_ok_docs = [row["doc_id"] for row in summary_rows if str(row.get("grobid_status") or "") == "ok"]
    outline_docs = [row["doc_id"] for row in summary_rows if int(row.get("outline_count") or 0) > 0]

    return {
        "selected_count": int(selected_count),
        "documents_processed": len(summary_rows),
        "fitz_available": bool(capabilities.get("fitz_available")),
        "pypdf_available": bool(capabilities.get("pypdf_available")),
        "docling_available": bool(capabilities.get("docling_available")),
        "grobid_configured": bool(capabilities.get("grobid", {}).get("configured")),
        "grobid_reachable": bool(capabilities.get("grobid", {}).get("reachable")),
        "readable_without_ocr_count": sum(1 for row in summary_rows if row.get("readable_without_ocr")),
        "unreadable_without_ocr_count": len(unreadable_docs),
        "unreadable_without_ocr_docs": unreadable_docs,
        "low_text_coverage_count": len(low_coverage_docs),
        "low_text_coverage_docs": low_coverage_docs,
        "cached_doc_count": len(cached_docs),
        "cached_doc_ids": cached_docs,
        "runtime_capability_mismatch_count": len(runtime_capability_mismatch_docs),
        "runtime_capability_mismatch_docs": runtime_capability_mismatch_docs,
        "fallback_activated_count": len(fallback_docs),
        "fallback_activated_docs": fallback_docs,
        "docling_success_like_count": len(docling_success_like_docs),
        "docling_success_count": len(docling_success_docs),
        "docling_partial_success_count": len(docling_partial_docs),
        "grobid_success_count": len(grobid_ok_docs),
        "outline_doc_count": len(outline_docs),
    }


def build_phase_b_assessment(
    summary_rows: List[Dict[str, Any]],
    capabilities: Dict[str, Any],
    selected_count: int,
) -> Dict[str, Any]:
    counts = compute_phase_b_counts(summary_rows, capabilities, selected_count)
    failures: List[str] = []
    warnings_list: List[str] = []
    infos: List[str] = []
    next_actions: List[str] = []

    if not counts["fitz_available"]:
        failures.append("PyMuPDF is unavailable. Phase B cannot satisfy the deterministic fallback contract.")
        next_actions.append("Install PyMuPDF in the notebook kernel and rerun Phase B.")
    if counts["documents_processed"] != counts["selected_count"]:
        failures.append(
            f"Only {counts['documents_processed']} of {counts['selected_count']} selected PDFs produced parser bundles."
        )
        next_actions.append("Inspect parser logs and diagnostics for missing bundle outputs.")
    if counts["unreadable_without_ocr_count"] > 0:
        failures.append(
            f"{counts['unreadable_without_ocr_count']} PDF(s) were not readable without OCR in a digital-only benchmark."
        )
        next_actions.append("Inspect the unreadable PDFs and confirm they are digitally extractable.")

    if not counts["pypdf_available"]:
        warnings_list.append("pypdf is unavailable in the current notebook runtime, so independent outline validation is missing.")
        next_actions.append("Install pypdf in the notebook kernel and inspect phase_b_runtime.json if the kernel path is unclear.")
    if not counts["docling_available"]:
        warnings_list.append("Docling is unavailable in the current notebook runtime.")
        next_actions.append("Install docling in the notebook kernel and inspect phase_b_runtime.json if the kernel path is unclear.")
    if counts["docling_success_like_count"] < max(1, counts["documents_processed"] // 2):
        warnings_list.append(
            "Docling succeeded only on a minority of documents, so Phase C will lean heavily on fallback structure signals."
        )
        next_actions.append("Review docling.json diagnostics and consider adjusting page limits or runtime setup.")
    if counts["low_text_coverage_count"] > 0:
        warnings_list.append(
            f"{counts['low_text_coverage_count']} PDF(s) had low extracted text coverage and may carry weak section evidence."
        )
        next_actions.append("Inspect pymupdf_pages.jsonl for low-coverage documents before Phase C.")
    if counts["fallback_activated_count"] > 0:
        warnings_list.append(
            f"Fallback parsing was activated for {counts['fallback_activated_count']} document(s)."
        )
    if counts["runtime_capability_mismatch_count"] > 0:
        warnings_list.append(
            "Some cached parser bundles were created under a different runtime capability profile than the current notebook session."
        )
        next_actions.append("Set force_rebuild=True for Phase B if you need a clean run under the current environment.")
    if not counts["grobid_configured"]:
        warnings_list.append("GROBID is not configured, so the scholarly enhancement lane is absent.")
        next_actions.append("Configure GROBID_URL or GROBID_BASE_URL if you want TEI structure recovery.")
    elif not counts["grobid_reachable"]:
        warnings_list.append("GROBID is configured but not reachable.")
        next_actions.append("Start or fix the GROBID service before rerunning Phase B.")

    if counts["outline_doc_count"] > 0:
        infos.append(f"{counts['outline_doc_count']} PDF(s) expose outline/bookmark structure already.")
    if counts["readable_without_ocr_count"] == counts["documents_processed"] and counts["documents_processed"] > 0:
        infos.append("All processed PDFs were readable without OCR.")

    if failures:
        status = "fail"
        quality_band = "degraded"
        can_continue = False
    elif warnings_list:
        status = "success_with_warnings"
        quality_band = "acceptable_with_issues"
        can_continue = True
    else:
        status = "success"
        quality_band = "strong"
        can_continue = True

    warnings_list = list(dict.fromkeys(warnings_list))
    next_actions = list(dict.fromkeys(next_actions))
    infos = list(dict.fromkeys(infos))

    return {
        "generated_at_utc": utc_now_iso(),
        "phase": "phase_b",
        "status": status,
        "quality_band": quality_band,
        "can_continue_to_next_phase": bool(can_continue),
        "failures": failures,
        "warnings": warnings_list,
        "info": infos,
        "recommended_next_actions": next_actions,
        "counts": counts,
    }


def build_qc_rows(
    summary_rows: List[Dict[str, Any]],
    capabilities: Dict[str, Any],
    selected_count: int,
    assessment: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    def qc_row(check: str, status: str, value: Any, expected: str, why: str, fix: str) -> Dict[str, Any]:
        return {
            "check": str(check),
            "status": str(status),
            "value": str(value),
            "expected": str(expected),
            "why": str(why),
            "fix": str(fix),
        }

    counts = (assessment or build_phase_b_assessment(summary_rows, capabilities, selected_count)).get("counts", {})
    low_coverage_docs = list(counts.get("low_text_coverage_docs") or [])
    unreadable_docs = list(counts.get("unreadable_without_ocr_docs") or [])
    docling_success_count = int(counts.get("docling_success_count") or 0)

    qc_rows = []
    qc_rows.append(
        qc_row(
            "documents_processed",
            "OK" if int(counts.get("documents_processed") or 0) == int(selected_count) else "FAIL",
            counts.get("documents_processed"),
            str(selected_count),
            "every selected PDF should emit a parser bundle",
            "inspect diagnostics.json for any missing document output",
        )
    )
    qc_rows.append(
        qc_row(
            "fitz_available",
            "OK" if bool(capabilities.get("fitz_available")) else "FAIL",
            bool(capabilities.get("fitz_available")),
            "True",
            "PyMuPDF is the deterministic fallback and text-coverage lane",
            "install PyMuPDF before rerunning Phase B",
        )
    )
    qc_rows.append(
        qc_row(
            "pypdf_available",
            "OK" if bool(capabilities.get("pypdf_available")) else "WARN",
            bool(capabilities.get("pypdf_available")),
            "True",
            "pypdf provides an independent metadata and outline lane",
            "install pypdf before rerunning Phase B",
        )
    )
    qc_rows.append(
        qc_row(
            "unreadable_without_ocr",
            "OK" if not unreadable_docs else "WARN",
            "none" if not unreadable_docs else ", ".join(unreadable_docs[:4]),
            "none",
            "this benchmark is restricted to digital PDFs with extractable text",
            "inspect diagnostics for any document flagged as unreadable",
        )
    )
    qc_rows.append(
        qc_row(
            "low_text_coverage_docs",
            "OK" if not low_coverage_docs else "WARN",
            "none" if not low_coverage_docs else ", ".join(low_coverage_docs[:4]),
            "none below 50% page text coverage",
            "low coverage usually indicates parser trouble or image-heavy pages",
            "inspect pymupdf_pages.jsonl and diagnostics for the affected PDFs",
        )
    )
    qc_rows.append(
        qc_row(
            "cache_runtime_mismatch",
            "OK" if int(counts.get("runtime_capability_mismatch_count") or 0) == 0 else "WARN",
            counts.get("runtime_capability_mismatch_count"),
            "0",
            "cached results from a different runtime can make the phase look healthier than the current kernel supports",
            "set force_rebuild=True for a clean run under the current environment",
        )
    )
    qc_rows.append(
        qc_row(
            "docling_success_count",
            "OK" if docling_success_count >= 1 else "WARN",
            docling_success_count,
            ">= 1 on a healthy environment, otherwise fallback path must carry",
            "Docling is the preferred structure-aware parser when it works",
            "inspect docling.json stdout/stderr and consider tuning page limits or environment setup",
        )
    )
    qc_rows.append(
        qc_row(
            "grobid_service",
            "OK" if bool(capabilities.get("grobid", {}).get("reachable")) else "WARN",
            capabilities.get("grobid", {}).get("status"),
            "alive when scholarly enhancement is configured",
            "GROBID is optional but valuable for article/report structure recovery",
            "start a GROBID service and set GROBID_URL or GROBID_BASE_URL",
        )
    )
    if assessment is not None:
        qc_rows.append(
            qc_row(
                "phase_b_status",
                "OK" if assessment.get("status") == "success" else ("WARN" if assessment.get("status") == "success_with_warnings" else "FAIL"),
                assessment.get("status"),
                "success or success_with_warnings",
                "this is the persisted phase-level verdict used to judge whether the stage is healthy enough to continue",
                "inspect phase_b_assessment.json and the document diagnostics before continuing",
            )
        )
    return qc_rows


def run_phase_b(
    run_ctx: Any,
    pdf_manifest: List[Dict[str, Any]],
    options: PhaseBOptions,
    *,
    stable_hash_fn: Optional[Callable[..., str]] = None,
    log_event_fn: Optional[Callable[..., Any]] = None,
    run_logger: Optional[Any] = None,
) -> Dict[str, Any]:
    options = options.normalized()
    capabilities = detect_capabilities(options.grobid_base_url)
    required_kernel_packages = required_phase_b_kernel_packages(options)
    missing_kernel_packages = missing_phase_b_kernel_packages(capabilities, options)

    parser_dir = ensure_dir(Path(run_ctx.artifacts.parser_dir))
    config_path = parser_dir / "phase_b_config.json"
    runtime_path = parser_dir / "phase_b_runtime.json"
    summary_path = parser_dir / "phase_b_summary.json"
    assessment_path = parser_dir / "phase_b_assessment.json"
    index_path = parser_dir / "parsed_document_bundles.jsonl"

    write_json_atomic(
        runtime_path,
        {
            "generated_at_utc": utc_now_iso(),
            "phase": "phase_b",
            "options": json_safe(options.__dict__),
            "required_kernel_packages": required_kernel_packages,
            "missing_kernel_packages": missing_kernel_packages,
            "capabilities": capabilities,
        },
    )

    config_payload = {
        "generated_at_utc": utc_now_iso(),
        "phase": "phase_b",
        "options": json_safe(options.__dict__),
        "capabilities": capabilities,
        "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
    }
    write_json_atomic(config_path, config_payload)

    selected_rows: List[Dict[str, Any]] = []
    include_doc_ids = set(options.include_doc_ids or [])
    exclude_doc_ids = set(options.exclude_doc_ids or [])

    for manifest_row in list(pdf_manifest or []):
        doc_id = compute_doc_id(manifest_row, stable_hash_fn=stable_hash_fn)
        if include_doc_ids and doc_id not in include_doc_ids:
            continue
        if doc_id in exclude_doc_ids:
            continue
        row = dict(manifest_row)
        row["doc_id"] = doc_id
        selected_rows.append(row)

    if options.doc_limit is not None:
        selected_rows = selected_rows[: int(options.doc_limit)]
    if not selected_rows:
        raise RuntimeError("Phase B selected zero PDFs after filtering. Adjust PhaseBOptions filters.")
    if run_logger is not None:
        run_logger.info(
            "Phase B runtime | python=%s | missing_kernel_packages=%s",
            capabilities.get("runtime", {}).get("python_executable"),
            ",".join(missing_kernel_packages) if missing_kernel_packages else "none",
        )
        run_logger.info(
            "Phase B started | selected=%s | force_rebuild=%s | fitz=%s | pypdf=%s | docling=%s | grobid=%s",
            len(selected_rows),
            options.force_rebuild,
            capabilities.get("fitz_available"),
            capabilities.get("pypdf_available"),
            capabilities.get("docling_available"),
            capabilities.get("grobid", {}).get("status"),
        )

    summary_rows: List[Dict[str, Any]] = []
    bundle_rows: List[Dict[str, Any]] = []

    for manifest_row in selected_rows:
        doc_id = str(manifest_row["doc_id"])
        source_path = Path(str(manifest_row["path"])).resolve()
        doc_dir = ensure_dir(parser_dir / doc_id)
        metadata_path = doc_dir / "metadata.json"
        diagnostics_path = doc_dir / "diagnostics.json"
        fitz_pages_path = doc_dir / "pymupdf_pages.jsonl"
        fitz_blocks_path = doc_dir / "pymupdf_blocks.jsonl"
        docling_path = doc_dir / "docling.json"
        grobid_summary_path = doc_dir / "grobid_summary.json"
        grobid_xml_path = doc_dir / "grobid.tei.xml"

        required_cache_paths = [metadata_path, diagnostics_path, fitz_pages_path, fitz_blocks_path, docling_path, grobid_summary_path]
        if (not options.force_rebuild) and all(p.exists() for p in required_cache_paths):
            cached_diag = read_json(diagnostics_path)
            cached_summary = dict(cached_diag.get("summary_row") or {})
            if cached_summary:
                runtime_snapshot = dict(cached_diag.get("runtime_capabilities_snapshot") or {})
                cached_options_snapshot = dict(cached_diag.get("phase_b_options_snapshot") or {})
                parser_statuses = dict(cached_diag.get("parser_statuses") or {})
                mismatch_fields: List[str] = []
                for field in ["fitz_available", "pypdf_available", "docling_available"]:
                    current_val = bool(capabilities.get(field))
                    cached_val = runtime_snapshot.get(field)
                    if isinstance(cached_val, bool):
                        if current_val != cached_val:
                            mismatch_fields.append(field)
                        continue
                    if field == "fitz_available":
                        cached_status = str(parser_statuses.get("fitz") or "")
                        if current_val and cached_status in {"", "unavailable"}:
                            mismatch_fields.append(field)
                        if (not current_val) and cached_status not in {"", "unavailable"}:
                            mismatch_fields.append(field)
                    if field == "pypdf_available":
                        cached_status = str(parser_statuses.get("pypdf") or "")
                        if current_val and cached_status in {"", "unavailable"}:
                            mismatch_fields.append(field)
                        if (not current_val) and cached_status not in {"", "unavailable"}:
                            mismatch_fields.append(field)
                    if field == "docling_available":
                        cached_status = str(parser_statuses.get("docling") or "")
                        docling_was_enabled = bool(cached_options_snapshot.get("try_docling", True))
                        if docling_was_enabled and current_val and cached_status in {"", "unavailable", "disabled"}:
                            mismatch_fields.append(field)
                        if (not current_val) and cached_status not in {"", "unavailable", "disabled"}:
                            mismatch_fields.append(field)
                cached_summary["cached"] = True
                cached_summary["cached_from_generated_at_utc"] = cached_diag.get("generated_at_utc")
                cached_summary["runtime_capability_mismatch"] = bool(mismatch_fields)
                cached_summary["runtime_capability_mismatch_fields"] = mismatch_fields
                summary_rows.append(cached_summary)
                bundle_rows.append(dict(cached_diag.get("bundle_row") or {}))
                if run_logger is not None:
                    run_logger.info(
                        "Phase B cached document | doc_id=%s | bundle_status=%s | docling=%s | grobid=%s | mismatch=%s",
                        doc_id,
                        cached_diag.get("bundle_status"),
                        cached_diag.get("parser_statuses", {}).get("docling"),
                        cached_diag.get("parser_statuses", {}).get("grobid"),
                        bool(mismatch_fields),
                    )
                if log_event_fn is not None:
                    log_event_fn(
                        run_ctx,
                        stage="phase_b",
                        event="document_reused_from_cache",
                        doc_id=doc_id,
                        bundle_status=cached_diag.get("bundle_status"),
                        docling_status=cached_diag.get("parser_statuses", {}).get("docling"),
                        grobid_status=cached_diag.get("parser_statuses", {}).get("grobid"),
                        runtime_capability_mismatch=bool(mismatch_fields),
                    )
                continue

        t0 = time.perf_counter()
        fitz_bundle = extract_fitz_bundle(source_path, min_page_words=options.min_page_words)
        pypdf_bundle = extract_pypdf_bundle(source_path)
        page_count = fitz_bundle.get("page_count") or pypdf_bundle.get("page_count") or manifest_row.get("page_count")
        fitz_pages = list(fitz_bundle.get("pages") or [])
        fitz_blocks = list(fitz_bundle.get("blocks") or [])
        pages_with_text = sum(1 for row in fitz_pages if row.get("has_text"))
        pages_with_substantive_text = sum(1 for row in fitz_pages if row.get("has_substantive_text"))
        total_chars = sum(int(row.get("char_len") or 0) for row in fitz_pages)
        pct_pages_with_text = round((pages_with_text / float(page_count)) * 100.0, 2) if page_count else None
        pct_pages_with_substantive_text = round((pages_with_substantive_text / float(page_count)) * 100.0, 2) if page_count else None
        readable_without_ocr = bool(total_chars >= int(options.min_doc_chars) and pages_with_substantive_text >= 1)
        ocr_required_or_unreadable = not readable_without_ocr
        outline_count = max(len(fitz_bundle.get("outline") or []), len(pypdf_bundle.get("outline") or []))
        page_count_agrees = (
            fitz_bundle.get("page_count") is None
            or pypdf_bundle.get("page_count") is None
            or int(fitz_bundle.get("page_count")) == int(pypdf_bundle.get("page_count"))
        )

        docling_bundle = extract_docling_bundle(source_path, page_count, options)
        docling_success = str(docling_bundle.get("status") or "") == "success"
        grobid_bundle = extract_grobid_bundle(source_path, manifest_row, page_count, options, capabilities)
        fallback_activated = bool(readable_without_ocr and not docling_success and str(fitz_bundle.get("status") or "") == "ok")
        bundle_status = "ok" if readable_without_ocr and str(fitz_bundle.get("status") or "") == "ok" else "needs_attention"
        elapsed_ms = round((time.perf_counter() - t0) * 1000.0, 3)

        metadata_payload = {
            "generated_at_utc": utc_now_iso(),
            "phase": "phase_b",
            "doc_id": doc_id,
            "label": manifest_row.get("label"),
            "source_path": str(source_path),
            "file_name": source_path.name,
            "sha256": manifest_row.get("sha256"),
            "size_mb": manifest_row.get("size_mb"),
            "page_count": page_count,
            "page_count_sources": {
                "phase_a_manifest": manifest_row.get("page_count"),
                "fitz": fitz_bundle.get("page_count"),
                "pypdf": pypdf_bundle.get("page_count"),
                "agree": bool(page_count_agrees),
            },
            "outline_counts": {
                "fitz": len(fitz_bundle.get("outline") or []),
                "pypdf": len(pypdf_bundle.get("outline") or []),
            },
            "text_coverage": {
                "pages_with_text": pages_with_text,
                "pages_with_substantive_text": pages_with_substantive_text,
                "percent_pages_with_text": pct_pages_with_text,
                "percent_pages_with_substantive_text": pct_pages_with_substantive_text,
                "total_chars": total_chars,
                "readable_without_ocr": bool(readable_without_ocr),
                "ocr_required_or_unreadable": bool(ocr_required_or_unreadable),
            },
            "fitz": {
                "status": fitz_bundle.get("status"),
                "metadata": fitz_bundle.get("metadata"),
                "outline": fitz_bundle.get("outline"),
                "error": fitz_bundle.get("error"),
            },
            "pypdf": {
                "status": pypdf_bundle.get("status"),
                "metadata": pypdf_bundle.get("metadata"),
                "outline": pypdf_bundle.get("outline"),
                "error": pypdf_bundle.get("error"),
            },
        }

        summary_row = {
            "doc_id": doc_id,
            "file_name": source_path.name,
            "page_count": page_count,
            "outline_count": outline_count,
            "pages_with_text_pct": pct_pages_with_text,
            "substantive_text_pct": pct_pages_with_substantive_text,
            "readable_without_ocr": bool(readable_without_ocr),
            "docling_status": docling_bundle.get("status"),
            "grobid_status": grobid_bundle.get("status"),
            "fallback_activated": bool(fallback_activated),
            "page_count_agrees": bool(page_count_agrees),
            "elapsed_ms": elapsed_ms,
            "cached": False,
            "cached_from_generated_at_utc": None,
            "runtime_capability_mismatch": False,
            "runtime_capability_mismatch_fields": [],
        }

        bundle_row = {
            "doc_id": doc_id,
            "source_path": str(source_path),
            "metadata_json": rel_to_run(Path(run_ctx.run_dir), metadata_path),
            "diagnostics_json": rel_to_run(Path(run_ctx.run_dir), diagnostics_path),
            "pymupdf_pages_jsonl": rel_to_run(Path(run_ctx.run_dir), fitz_pages_path),
            "pymupdf_blocks_jsonl": rel_to_run(Path(run_ctx.run_dir), fitz_blocks_path),
            "docling_json": rel_to_run(Path(run_ctx.run_dir), docling_path),
            "grobid_summary_json": rel_to_run(Path(run_ctx.run_dir), grobid_summary_path),
            "grobid_tei_xml": rel_to_run(Path(run_ctx.run_dir), grobid_xml_path) if grobid_bundle.get("xml_text") else None,
            "bundle_status": bundle_status,
        }

        diagnostics_payload = {
            "generated_at_utc": utc_now_iso(),
            "phase": "phase_b",
            "doc_id": doc_id,
            "bundle_status": bundle_status,
            "summary_row": summary_row,
            "bundle_row": bundle_row,
            "readable_without_ocr": bool(readable_without_ocr),
            "ocr_required_or_unreadable": bool(ocr_required_or_unreadable),
            "fallback_activated": bool(fallback_activated),
            "page_count_agrees": bool(page_count_agrees),
            "parser_statuses": {
                "fitz": fitz_bundle.get("status"),
                "pypdf": pypdf_bundle.get("status"),
                "docling": docling_bundle.get("status"),
                "grobid": grobid_bundle.get("status"),
            },
            "runtime_capabilities_snapshot": {
                "fitz_available": bool(capabilities.get("fitz_available")),
                "pypdf_available": bool(capabilities.get("pypdf_available")),
                "docling_available": bool(capabilities.get("docling_available")),
                "grobid_configured": bool(capabilities.get("grobid", {}).get("configured")),
                "grobid_reachable": bool(capabilities.get("grobid", {}).get("reachable")),
            },
            "phase_b_options_snapshot": json_safe(options.__dict__),
            "artifact_paths": bundle_row,
        }

        write_json_atomic(metadata_path, metadata_payload)
        write_jsonl_rows(fitz_pages_path, fitz_pages)
        write_jsonl_rows(fitz_blocks_path, fitz_blocks)
        write_json_atomic(docling_path, docling_bundle)
        write_json_atomic(grobid_summary_path, {k: v for k, v in grobid_bundle.items() if k != "xml_text"})
        if grobid_bundle.get("xml_text"):
            grobid_xml_path.write_text(str(grobid_bundle["xml_text"]), encoding="utf-8")
        elif grobid_xml_path.exists() and options.force_rebuild:
            grobid_xml_path.unlink()
        write_json_atomic(diagnostics_path, diagnostics_payload)

        summary_rows.append(summary_row)
        bundle_rows.append(bundle_row)

        if run_logger is not None:
            run_logger.info(
                "Phase B parsed document | doc_id=%s | page_count=%s | fitz=%s | pypdf=%s | docling=%s | grobid=%s | fallback=%s | elapsed_ms=%s",
                doc_id,
                page_count,
                fitz_bundle.get("status"),
                pypdf_bundle.get("status"),
                docling_bundle.get("status"),
                grobid_bundle.get("status"),
                bool(fallback_activated),
                elapsed_ms,
            )
        if log_event_fn is not None:
            log_event_fn(
                run_ctx,
                stage="phase_b",
                event="document_parsed",
                doc_id=doc_id,
                source_path=str(source_path),
                page_count=page_count,
                fitz_status=fitz_bundle.get("status"),
                pypdf_status=pypdf_bundle.get("status"),
                docling_status=docling_bundle.get("status"),
                grobid_status=grobid_bundle.get("status"),
                readable_without_ocr=bool(readable_without_ocr),
                fallback_activated=bool(fallback_activated),
                elapsed_ms=elapsed_ms,
            )

    assessment = build_phase_b_assessment(summary_rows, capabilities, len(selected_rows))
    qc_rows = build_qc_rows(summary_rows, capabilities, len(selected_rows), assessment=assessment)

    write_json_atomic(
        summary_path,
        {
            "generated_at_utc": utc_now_iso(),
            "run_id": run_ctx.run_id,
            "phase": "phase_b",
            "options": json_safe(options.__dict__),
            "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
            "capabilities": capabilities,
            "assessment": assessment,
            "qc_rows": qc_rows,
            "documents": summary_rows,
            "artifacts": bundle_rows,
        },
    )
    write_json_atomic(
        assessment_path,
        {
            "generated_at_utc": utc_now_iso(),
            "run_id": run_ctx.run_id,
            "phase": "phase_b",
            "assessment": assessment,
            "qc_rows": qc_rows,
            "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
            "summary_path": rel_to_run(Path(run_ctx.run_dir), summary_path),
            "index_path": rel_to_run(Path(run_ctx.run_dir), index_path),
        },
    )
    write_jsonl_rows(index_path, bundle_rows)
    if run_logger is not None:
        run_logger.info(
            "Phase B completed | status=%s | quality=%s | processed=%s | cached=%s | fallback=%s | warnings=%s | failures=%s",
            assessment.get("status"),
            assessment.get("quality_band"),
            assessment.get("counts", {}).get("documents_processed"),
            assessment.get("counts", {}).get("cached_doc_count"),
            assessment.get("counts", {}).get("fallback_activated_count"),
            len(assessment.get("warnings") or []),
            len(assessment.get("failures") or []),
        )

    metrics_update = {
        "initialized_at_utc": utc_now_iso(),
        "document_count": len(summary_rows),
        "readable_document_count": sum(1 for row in summary_rows if row.get("readable_without_ocr")),
        "docling_success_count": sum(1 for row in summary_rows if str(row.get("docling_status") or "") == "success"),
        "docling_success_like_count": sum(1 for row in summary_rows if _docling_success_like(row.get("docling_status"))),
        "grobid_success_count": sum(1 for row in summary_rows if str(row.get("grobid_status") or "") == "ok"),
        "grobid_reachable": bool(capabilities.get("grobid", {}).get("reachable")),
        "cached_doc_count": sum(1 for row in summary_rows if row.get("cached")),
        "runtime_capability_mismatch_count": sum(1 for row in summary_rows if row.get("runtime_capability_mismatch")),
        "status": assessment.get("status"),
        "quality_band": assessment.get("quality_band"),
        "can_continue_to_next_phase": assessment.get("can_continue_to_next_phase"),
        "warning_count": len(assessment.get("warnings") or []),
        "failure_count": len(assessment.get("failures") or []),
        "assessment_path": rel_to_run(Path(run_ctx.run_dir), assessment_path),
    }

    return {
        "config_path": config_path,
        "runtime_path": runtime_path,
        "summary_path": summary_path,
        "assessment_path": assessment_path,
        "index_path": index_path,
        "capabilities": capabilities,
        "required_kernel_packages": required_kernel_packages,
        "missing_kernel_packages": missing_kernel_packages,
        "summary_rows": summary_rows,
        "bundle_rows": bundle_rows,
        "metrics_update": metrics_update,
        "assessment": assessment,
        "qc_rows": qc_rows,
        "selected_count": len(selected_rows),
    }


In [ ]:
# Phase B.1 - Run parser bundles and print QC summary

phase_b_options = PhaseBOptions(
    force_rebuild=True,
    doc_limit=None,
    include_doc_ids=[],
    exclude_doc_ids=[],
    min_page_words=20,
    min_doc_chars=200,
    try_docling=True,
    docling_page_limit=200,
    try_grobid=True,
    grobid_page_limit=200,
    grobid_base_url=(os.getenv("GROBID_URL") or os.getenv("GROBID_BASE_URL") or "").strip(),
    grobid_process_path="/api/processFulltextDocument",
    grobid_timeout_sec=120,
    grobid_consolidate_header=0,
    grobid_consolidate_citations=0,
    grobid_include_raw_citations=0,
)

phase_b_logger = setup_run_logger(RUN_CONTEXT)

with stage_timer(RUN_CONTEXT, "phase_b"):
    phase_b_result = run_phase_b(
        RUN_CONTEXT,
        PDF_MANIFEST,
        phase_b_options,
        stable_hash_fn=stable_hash,
        log_event_fn=log_event,
        run_logger=phase_b_logger,
    )
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_b", {}).update(phase_b_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_b_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))

PHASE_B_RESULT = phase_b_result
PHASE_B_SUMMARY = phase_b_result["summary_rows"]
PHASE_B_BUNDLES = phase_b_result["bundle_rows"]

docling_success_count = sum(1 for row in PHASE_B_SUMMARY if str(row.get("docling_status") or "") == "success")
grobid_success_count = sum(1 for row in PHASE_B_SUMMARY if str(row.get("grobid_status") or "") == "ok")
fallback_activated_docs = sum(1 for row in PHASE_B_SUMMARY if row.get("fallback_activated"))

print_section("Phase B - Parser Capabilities")
print_kv(
    {
        "fitz_available": phase_b_result["capabilities"].get("fitz_available"),
        "pypdf_available": phase_b_result["capabilities"].get("pypdf_available"),
        "docling_available": phase_b_result["capabilities"].get("docling_available"),
        "python_executable": phase_b_result["capabilities"].get("runtime", {}).get("python_executable"),
        "grobid_status": phase_b_result["capabilities"].get("grobid", {}).get("status"),
        "selected_documents": phase_b_result["selected_count"],
        "docling_success_count": docling_success_count,
        "grobid_success_count": grobid_success_count,
    }
)

print_section("Phase B - What Happened")
print_kv(
    {
        "phase_b_config_json": phase_b_rel(phase_b_result["config_path"]),
        "phase_b_runtime_json": phase_b_rel(phase_b_result["runtime_path"]),
        "phase_b_summary_json": phase_b_rel(phase_b_result["summary_path"]),
        "phase_b_assessment_json": phase_b_rel(phase_b_result["assessment_path"]),
        "parsed_document_bundles_jsonl": phase_b_rel(phase_b_result["index_path"]),
        "documents_processed": len(PHASE_B_SUMMARY),
        "readable_without_ocr": sum(1 for row in PHASE_B_SUMMARY if row.get("readable_without_ocr")),
        "fallback_activated_docs": fallback_activated_docs,
        "missing_kernel_packages": ", ".join(phase_b_result.get("missing_kernel_packages") or []) or "none",
        "phase_status": phase_b_result["assessment"].get("status"),
        "quality_band": phase_b_result["assessment"].get("quality_band"),
    }
)

print_section("Phase B - Document Summary")
print_table(
    PHASE_B_SUMMARY,
    columns=[
        "doc_id",
        "file_name",
        "page_count",
        "outline_count",
        "pages_with_text_pct",
        "docling_status",
        "grobid_status",
        "fallback_activated",
    ],
    max_rows=20,
    max_col_width=44,
)

print_section("Phase B - Artifact Preview")
print_table(
    PHASE_B_BUNDLES,
    columns=["doc_id", "metadata_json", "pymupdf_blocks_jsonl", "docling_json", "grobid_tei_xml", "bundle_status"],
    max_rows=20,
    max_col_width=44,
)

print_section("Phase B - Assessment")
print_kv(
    {
        "status": phase_b_result["assessment"].get("status"),
        "quality_band": phase_b_result["assessment"].get("quality_band"),
        "can_continue": phase_b_result["assessment"].get("can_continue_to_next_phase"),
        "warning_count": len(phase_b_result["assessment"].get("warnings") or []),
        "failure_count": len(phase_b_result["assessment"].get("failures") or []),
        "cached_doc_count": phase_b_result["assessment"].get("counts", {}).get("cached_doc_count"),
        "runtime_capability_mismatch_count": phase_b_result["assessment"].get("counts", {}).get("runtime_capability_mismatch_count"),
    }
)

print_section("Phase B - QC")
print_table(
    phase_b_result["qc_rows"],
    columns=["check", "status", "value", "expected", "why", "fix"],
    max_rows=20,
    max_col_width=46,
)


---

## Phase C - Canonical section and passage normalization


In [ ]:
# Phase C.0 - Canonical document, section, and passage normalization helpers

import json
import math
import re
import sys
import unicodedata
import xml.etree.ElementTree as ET
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


@dataclass
class PhaseCOptions:
    force_rebuild: bool = False
    doc_limit: Optional[int] = None
    include_doc_ids: Optional[List[str]] = None
    exclude_doc_ids: Optional[List[str]] = None
    prefer_outline: bool = True
    use_docling: bool = True
    use_grobid: bool = True
    use_heuristic_headings: bool = True
    heuristic_heading_min_words: int = 1
    heuristic_heading_max_words: int = 18
    heuristic_heading_max_chars: int = 160
    repeated_heading_page_threshold: int = 3
    min_section_chars: int = 120
    min_section_words: int = 20
    min_section_coverage_pct_warn: float = 70.0
    long_doc_page_threshold: int = 40
    passage_target_words: int = 180
    passage_max_words: int = 260
    passage_min_words: int = 70
    synthesize_front_matter: bool = True
    synthesize_document_body: bool = True

    def normalized(self) -> "PhaseCOptions":
        return PhaseCOptions(
            force_rebuild=bool(self.force_rebuild),
            doc_limit=None if self.doc_limit is None else int(self.doc_limit),
            include_doc_ids=[str(x).strip() for x in (self.include_doc_ids or []) if str(x).strip()],
            exclude_doc_ids=[str(x).strip() for x in (self.exclude_doc_ids or []) if str(x).strip()],
            prefer_outline=bool(self.prefer_outline),
            use_docling=bool(self.use_docling),
            use_grobid=bool(self.use_grobid),
            use_heuristic_headings=bool(self.use_heuristic_headings),
            heuristic_heading_min_words=int(self.heuristic_heading_min_words),
            heuristic_heading_max_words=int(self.heuristic_heading_max_words),
            heuristic_heading_max_chars=int(self.heuristic_heading_max_chars),
            repeated_heading_page_threshold=int(self.repeated_heading_page_threshold),
            min_section_chars=int(self.min_section_chars),
            min_section_words=int(self.min_section_words),
            min_section_coverage_pct_warn=float(self.min_section_coverage_pct_warn),
            long_doc_page_threshold=int(self.long_doc_page_threshold),
            passage_target_words=int(self.passage_target_words),
            passage_max_words=int(self.passage_max_words),
            passage_min_words=int(self.passage_min_words),
            synthesize_front_matter=bool(self.synthesize_front_matter),
            synthesize_document_body=bool(self.synthesize_document_body),
        )


def read_jsonl_rows(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows: List[Dict[str, Any]] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def ascii_fold(text: Any) -> str:
    return unicodedata.normalize("NFKD", str(text or "")).encode("ascii", "ignore").decode("ascii")


def normalize_heading_display(text: Any) -> str:
    s = clean_text(text)
    s = s.replace("•", " ")
    s = s.replace("·", " ")
    s = s.replace("ﬁ", "fi")
    s = s.replace("ﬂ", "fl")
    s = re.sub(r"\s+", " ", s)
    return s.strip(" :-\t\n\r")


def strip_heading_numbering(text: Any) -> str:
    s = normalize_heading_display(text)
    s = re.sub(r"^(?:chapter|part)\s+[ivxlcdm0-9]+(?:\s*[:.\-])?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^(?:\d+(?:\.\d+){0,4}|[ivxlcdm]+|[A-Z])(?:[.)]|\s+-)\s*", "", s)
    return s.strip()


def normalize_heading_key(text: Any) -> str:
    s = ascii_fold(strip_heading_numbering(text)).lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return s.strip()


def heading_key_without_numbers(text: Any) -> str:
    s = normalize_heading_key(text)
    s = re.sub(r"\b\d+\b", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def has_heading_numbering(text: Any) -> bool:
    s = normalize_heading_display(text)
    return bool(
        re.match(r"^(?:chapter|part)\s+[ivxlcdm0-9]+\b", s, flags=re.IGNORECASE)
        or re.match(r"^\d+(?:\.\d+){0,4}\b", s)
        or re.match(r"^[ivxlcdm]+[.)]\s+", s, flags=re.IGNORECASE)
    )


def alnum_ratio(text: Any) -> float:
    s = ascii_fold(text)
    visible = re.sub(r"\s+", "", s)
    if not visible:
        return 0.0
    return sum(ch.isalnum() for ch in visible) / max(1, len(visible))


def has_math_unicode_signal(text: Any) -> bool:
    for ch in str(text or ""):
        try:
            name = unicodedata.name(ch)
        except Exception:
            continue
        if "MATHEMATICAL" in name or "DOUBLE-STRUCK" in name:
            return True
    return False


def infer_heading_level(title: str, source: str, level_hint: Optional[int] = None) -> int:
    if level_hint is not None:
        try:
            return max(1, int(level_hint))
        except Exception:
            pass
    s = normalize_heading_display(title)
    match = re.match(r"^(\d+(?:\.\d+){0,4})\b", s)
    if match:
        return max(1, len(match.group(1).split(".")))
    if re.match(r"^(?:chapter|part)\b", s, flags=re.IGNORECASE):
        return 1
    if source.startswith("heuristic") and s.isupper() and count_words(s) <= 6:
        return 1
    return 1


def infer_language_guess(text: str) -> str:
    sample = " " + ascii_fold(text).lower() + " "
    english_hits = sum(sample.count(token) for token in [" the ", " and ", " of ", " is ", " for ", " with "])
    german_hits = sum(sample.count(token) for token in [" der ", " die ", " das ", " und ", " mit ", " nicht "])
    if german_hits > english_hits * 1.2:
        return "de"
    if english_hits > 0:
        return "en"
    return "unknown"


def infer_doc_type_guess(metadata: Dict[str, Any]) -> str:
    file_name = str(metadata.get("file_name") or "")
    page_count = int(metadata.get("page_count") or 0)
    outline_count = int((metadata.get("outline_counts") or {}).get("fitz") or 0)
    title = normalize_heading_display((metadata.get("fitz") or {}).get("metadata", {}).get("title") or file_name)
    blob = " ".join(
        [
            file_name,
            title,
            json.dumps((metadata.get("fitz") or {}).get("metadata", {}), ensure_ascii=False),
            json.dumps((metadata.get("pypdf") or {}).get("metadata", {}), ensure_ascii=False),
        ]
    ).lower()
    if page_count >= 120:
        return "book_or_long_report"
    if any(token in blob for token in ["journal", "doi", "accepted manuscript", "abstract"]):
        return "scholarly_article"
    if outline_count >= 15 or any(token in blob for token in ["contents", "preface", "chapter"]):
        return "report_or_book"
    return "paper_or_report"


SECTION_TYPE_PATTERNS: List[Tuple[str, List[str]]] = [
    ("table_of_contents", [r"^contents$", r"^table of contents$", r"^list of figures$", r"^list of tables$"]),
    ("abstract", [r"^abstract$"]),
    ("introduction", [r"^introduction$", r"^1 introduction$"]),
    (
        "background",
        [
            r"^background$",
            r"^literature review$",
            r"^theoretical background$",
            r"^related work$",
            r"^conceptual background$",
            r"^conceptual framework$",
            r"^theoretical framework$",
            r"^theory and hypotheses$",
        ],
    ),
    (
        "methods",
        [
            r"^methods?$",
            r"^methodology$",
            r"^research design$",
            r"^data and methods$",
            r"^materials and methods$",
            r"^data collection$",
            r"^research methods?$",
            r"^empirical setting$",
            r"^study design$",
            r"^measures?$",
            r"^main measures$",
            r"^measurement$",
            r"^measurement model$",
            r"^sample and procedures?$",
            r"^variables?$",
        ],
    ),
    ("results", [r"^results?$", r"^analysis and results$", r"^findings$", r"^matching and findings$", r"^empirical results$", r"^analysis$", r"^empirical analysis$", r"^results and discussion$", r"^data analysis$"]),
    ("discussion", [r"^discussion$", r"^discussion and implications$", r"^implications$", r"^theoretical implications$", r"^managerial implications$", r"^practical implications$", r"^limitations$", r"^limitations and future research$"]),
    ("conclusion", [r"^conclusion$", r"^conclusions$", r"^general discussion and conclusions?$", r"^summary and conclusion$", r"^future research$", r"^concluding remarks$", r"^summary$", r"^general discussion$"]),
    ("references", [r"^references$", r"^bibliography$", r"^works cited$"]),
    ("appendix", [r"^appendix(?:\s+[a-z0-9]+)?$", r"^appendices$", r"^supplement(?:ary)? materials?$", r"^supplementary information$"]),
    ("index", [r"^index$", r"^subject index$", r"^author index$", r"^glossary$", r"^nomenclature$", r"^abbreviations?$"]),
    ("acknowledgements", [r"^acknowledg?ments?$", r"^author contributions$", r"^funding$"]),
]


def classify_section_type(title: str) -> str:
    key = normalize_heading_key(title)
    if not key:
        return "body_other"
    for section_type, patterns in SECTION_TYPE_PATTERNS:
        for pattern in patterns:
            if re.match(pattern, key, flags=re.IGNORECASE):
                return section_type
    if key in {"front matter", "title page", "copyright page", "preface", "foreword", "list of contributors", "about the authors"}:
        return "front_matter"
    return "body_other"


KNOWN_NOISE_HEADING_PATTERNS = [
    r"^accepted manuscript$",
    r"^research article$",
    r"^please cite this article",
    r"^doi$",
    r"^pii$",
    r"^received date$",
    r"^revised date$",
    r"^accepted date$",
    r"^to appear in$",
    r"^keywords$",
    r"^reference$",
    r"^figure \d+",
    r"^table \d+",
    r"^model(?: \d+)?$",
    r"^sample$",
    r"^dv[: ]",
    r"^products with$",
]


def is_probable_noise_heading(
    title: str,
    *,
    source: str,
    repeated_pages: int,
    repeated_page_threshold: int,
    doc_title_key: str,
) -> Tuple[bool, str]:
    display = normalize_heading_display(title)
    key = normalize_heading_key(display)
    words = count_words(display)
    if re.search(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", str(title or "")):
        return True, "control_characters"
    if has_math_unicode_signal(title):
        return True, "mathematical_unicode_signal"
    if not key:
        return True, "empty_after_normalization"
    if len(display) < 3:
        return True, "too_short"
    if len(display) > 180:
        return True, "too_long"
    if words > 24:
        return True, "too_many_words"
    if alnum_ratio(display) < 0.45 and not has_heading_numbering(display):
        return True, "low_alnum_ratio"
    if repeated_pages >= int(repeated_page_threshold) and not source.startswith("outline"):
        return True, "repeated_page_header"
    if re.search(r"(->|<-|=|<|>|@|\bvol\.\b|\bno\.\b|issn|isbn|doi:)", display, flags=re.IGNORECASE):
        return True, "metadata_or_table_signal"
    if re.search(r"\*{2,}|={2,}|%|±|β|γ|δ", display):
        return True, "formula_or_numeric_signal"
    if key and sum(ch.isdigit() for ch in display) >= max(3, math.ceil(len(display) * 0.18)) and not has_heading_numbering(display):
        return True, "digit_heavy"
    for pattern in KNOWN_NOISE_HEADING_PATTERNS:
        if re.match(pattern, key, flags=re.IGNORECASE):
            return True, "known_noise_pattern"
    if doc_title_key and not source.startswith("outline") and title_similarity(key, doc_title_key) >= 0.9:
        return True, "document_title_repeat"
    return False, ""


def title_similarity(a: str, b: str) -> float:
    a_key = normalize_heading_key(a)
    b_key = normalize_heading_key(b)
    if not a_key or not b_key:
        return 0.0
    if a_key == b_key:
        return 1.0
    a_tokens = set(a_key.split())
    b_tokens = set(b_key.split())
    overlap = len(a_tokens & b_tokens) / max(1, len(a_tokens | b_tokens))
    seq = SequenceMatcher(None, a_key, b_key).ratio()
    return max(overlap, seq)


def block_text_variants(text: str) -> List[str]:
    raw = clean_text(text)
    if not raw:
        return []
    variants = [raw]
    parts = [normalize_heading_display(part) for part in re.split(r"\n+", raw) if normalize_heading_display(part)]
    variants.extend(parts[:4])
    return list(dict.fromkeys([v for v in variants if v]))


def build_block_index(block_rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    ordered = sorted(block_rows, key=lambda row: (int(row.get("page") or 0), int(row.get("block_index") or 0)))
    pages: Dict[int, List[Dict[str, Any]]] = {}
    repeated: Dict[str, set] = {}
    repeated_loose: Dict[str, set] = {}
    for abs_idx, row in enumerate(ordered):
        row["abs_block_index"] = abs_idx
        page = int(row.get("page") or 0)
        pages.setdefault(page, []).append(row)
        key = normalize_heading_key(row.get("text"))
        loose_key = heading_key_without_numbers(row.get("text"))
        words = count_words(row.get("text"))
        if key and 1 <= words <= 8 and len(normalize_heading_display(row.get("text"))) <= 80:
            repeated.setdefault(key, set()).add(page)
        if loose_key and 2 <= len(loose_key.split()) <= 12 and len(normalize_heading_display(row.get("text"))) <= 120:
            repeated_loose.setdefault(loose_key, set()).add(page)
    return {
        "ordered_blocks": ordered,
        "blocks_by_page": pages,
        "repeated_heading_pages": {key: len(page_set) for key, page_set in repeated.items()},
        "repeated_heading_pages_wo_numbers": {key: len(page_set) for key, page_set in repeated_loose.items()},
        "total_block_chars": sum(int(row.get("char_len") or len(row.get("text") or "")) for row in ordered),
        "total_block_words": sum(int(row.get("word_count") or count_words(row.get("text"))) for row in ordered),
    }


def load_phase_b_bundle(run_ctx: Any, bundle_row: Dict[str, Any]) -> Dict[str, Any]:
    run_dir = Path(run_ctx.run_dir)

    def resolve(rel_path: Optional[str]) -> Optional[Path]:
        if not rel_path:
            return None
        path = Path(rel_path)
        if not path.is_absolute():
            path = run_dir / path
        return path

    metadata_path = resolve(bundle_row.get("metadata_json"))
    pages_path = resolve(bundle_row.get("pymupdf_pages_jsonl"))
    blocks_path = resolve(bundle_row.get("pymupdf_blocks_jsonl"))
    docling_path = resolve(bundle_row.get("docling_json"))
    diagnostics_path = resolve(bundle_row.get("diagnostics_json"))
    grobid_summary_path = resolve(bundle_row.get("grobid_summary_json"))
    grobid_tei_path = resolve(bundle_row.get("grobid_tei_xml"))
    return {
        "bundle_row": bundle_row,
        "metadata": read_json(metadata_path) if metadata_path and metadata_path.exists() else {},
        "pages": read_jsonl_rows(pages_path) if pages_path and pages_path.exists() else [],
        "blocks": read_jsonl_rows(blocks_path) if blocks_path and blocks_path.exists() else [],
        "docling": read_json(docling_path) if docling_path and docling_path.exists() else {},
        "diagnostics": read_json(diagnostics_path) if diagnostics_path and diagnostics_path.exists() else {},
        "grobid_summary": read_json(grobid_summary_path) if grobid_summary_path and grobid_summary_path.exists() else {},
        "grobid_tei_text": grobid_tei_path.read_text(encoding="utf-8", errors="ignore") if grobid_tei_path and grobid_tei_path.exists() else "",
        "paths": {
            "metadata": metadata_path,
            "pages": pages_path,
            "blocks": blocks_path,
            "docling": docling_path,
            "diagnostics": diagnostics_path,
            "grobid_summary": grobid_summary_path,
            "grobid_tei": grobid_tei_path,
        },
    }


def extract_outline_proposals(metadata: Dict[str, Any]) -> List[Dict[str, Any]]:
    proposals: List[Dict[str, Any]] = []
    seen = set()
    for source_name in ["fitz", "pypdf"]:
        outline = ((metadata.get(source_name) or {}).get("outline") or [])
        for idx, item in enumerate(outline):
            title = normalize_heading_display(item.get("title"))
            page = item.get("page")
            level = item.get("level")
            key = (normalize_heading_key(title), int(page or 0), int(level or 0), source_name)
            if key in seen or not title:
                continue
            seen.add(key)
            proposals.append(
                {
                    "proposal_id": f"outline_{source_name}_{idx}",
                    "source": f"outline_{source_name}",
                    "title": title,
                    "page": int(page) if page else None,
                    "level_hint": int(level) if level else None,
                    "source_priority": 100 if source_name == "fitz" else 95,
                    "raw": item,
                }
            )
    return proposals


def extract_docling_proposals(docling_bundle: Dict[str, Any]) -> List[Dict[str, Any]]:
    proposals: List[Dict[str, Any]] = []
    doc = (docling_bundle or {}).get("document") or {}
    texts = doc.get("texts") or []
    for idx, item in enumerate(texts):
        if str(item.get("label") or "") != "section_header":
            continue
        title = normalize_heading_display(item.get("text"))
        prov = item.get("prov") or []
        page = None
        if prov:
            page = prov[0].get("page_no")
        proposals.append(
            {
                "proposal_id": f"docling_{idx}",
                "source": "docling",
                "title": title,
                "page": int(page) if page else None,
                "level_hint": None,
                "source_priority": 80,
                "raw": item,
            }
        )
    return proposals


def extract_grobid_proposals(grobid_tei_text: str) -> List[Dict[str, Any]]:
    proposals: List[Dict[str, Any]] = []
    xml_text = str(grobid_tei_text or "").strip()
    if not xml_text:
        return proposals
    try:
        root = ET.fromstring(xml_text)
    except Exception:
        return proposals
    idx = 0
    for head in root.findall(".//{*}head"):
        title = normalize_heading_display("".join(head.itertext()))
        if not title:
            continue
        proposals.append(
            {
                "proposal_id": f"grobid_{idx}",
                "source": "grobid",
                "title": title,
                "page": None,
                "level_hint": None,
                "source_priority": 90,
                "raw": {"tag": head.tag},
            }
        )
        idx += 1
    return proposals


def looks_like_heading_block(row: Dict[str, Any], options: PhaseCOptions) -> bool:
    text = normalize_heading_display(row.get("text"))
    words = count_words(text)
    if not text:
        return False
    if words < int(options.heuristic_heading_min_words) or words > int(options.heuristic_heading_max_words):
        return False
    if len(text) > int(options.heuristic_heading_max_chars):
        return False
    if alnum_ratio(text) < 0.5 and not has_heading_numbering(text):
        return False
    if text.endswith(".") and words > 6:
        return False
    titlecase_hits = sum(1 for part in text.split() if part[:1].isupper())
    if titlecase_hits >= max(1, math.ceil(words * 0.5)):
        return True
    if has_heading_numbering(text):
        return True
    if text.isupper() and words <= 8:
        return True
    if classify_section_type(text) != "body_other":
        return True
    return False


def extract_heuristic_proposals(block_rows: List[Dict[str, Any]], options: PhaseCOptions) -> List[Dict[str, Any]]:
    proposals: List[Dict[str, Any]] = []
    for row in block_rows:
        text = normalize_heading_display(row.get("text"))
        if not looks_like_heading_block(row, options):
            continue
        proposals.append(
            {
                "proposal_id": f"heur_{row.get('page')}_{row.get('block_index')}",
                "source": "heuristic_block",
                "title": text,
                "page": int(row.get("page") or 0) or None,
                "level_hint": infer_heading_level(text, "heuristic_block"),
                "source_priority": 60,
                "raw": {"page": row.get("page"), "block_index": row.get("block_index")},
                "anchor_page": row.get("page"),
                "anchor_block_index": row.get("block_index"),
                "anchor_abs_block_index": row.get("abs_block_index"),
                "anchor_method": "direct_block",
                "anchor_confidence": 1.0,
            }
        )
    return proposals


def score_anchor_match(title: str, block_text: str) -> float:
    best = 0.0
    title_key = normalize_heading_key(title)
    if not title_key:
        return 0.0
    for variant in block_text_variants(block_text):
        variant_key = normalize_heading_key(variant)
        if not variant_key:
            continue
        if variant_key == title_key:
            return 1.0
        if variant_key.startswith(title_key) or title_key.startswith(variant_key):
            best = max(best, 0.93)
        else:
            best = max(best, title_similarity(title_key, variant_key))
    return best


def anchor_proposal(proposal: Dict[str, Any], block_index: Dict[str, Any]) -> Dict[str, Any]:
    if proposal.get("anchor_abs_block_index") is not None:
        return proposal
    title = proposal.get("title")
    page = proposal.get("page")
    search_blocks: List[Dict[str, Any]] = []
    if page is not None:
        search_blocks.extend(block_index["blocks_by_page"].get(int(page), []))
        if not search_blocks:
            search_blocks.extend(block_index["blocks_by_page"].get(int(page) - 1, []))
            search_blocks.extend(block_index["blocks_by_page"].get(int(page) + 1, []))
    if not search_blocks:
        search_blocks = block_index["ordered_blocks"]
    best_row = None
    best_score = 0.0
    for row in search_blocks:
        score = score_anchor_match(title, row.get("text"))
        if score > best_score:
            best_score = score
            best_row = row
    anchored = dict(proposal)
    if best_row is not None and best_score >= 0.78:
        anchored["anchor_page"] = int(best_row.get("page") or 0) or None
        anchored["anchor_block_index"] = int(best_row.get("block_index") or 0)
        anchored["anchor_abs_block_index"] = int(best_row.get("abs_block_index") or 0)
        anchored["anchor_method"] = "block_text_match"
        anchored["anchor_confidence"] = round(float(best_score), 3)
        return anchored
    if page is not None and block_index["blocks_by_page"].get(int(page)):
        fallback = block_index["blocks_by_page"][int(page)][0]
        anchored["anchor_page"] = int(fallback.get("page") or 0) or None
        anchored["anchor_block_index"] = int(fallback.get("block_index") or 0)
        anchored["anchor_abs_block_index"] = int(fallback.get("abs_block_index") or 0)
        anchored["anchor_method"] = "page_start_fallback"
        anchored["anchor_confidence"] = round(float(best_score), 3)
        return anchored
    return anchored


def filter_and_anchor_proposals(proposals: List[Dict[str, Any]], *, block_index: Dict[str, Any], metadata: Dict[str, Any], options: PhaseCOptions) -> Dict[str, Any]:
    doc_title = normalize_heading_display((metadata.get("fitz") or {}).get("metadata", {}).get("title") or metadata.get("label") or metadata.get("file_name"))
    doc_title_key = normalize_heading_key(doc_title)
    repeated_pages = block_index.get("repeated_heading_pages") or {}
    repeated_pages_loose = block_index.get("repeated_heading_pages_wo_numbers") or {}
    proposal_rows: List[Dict[str, Any]] = []
    accepted: List[Dict[str, Any]] = []
    seen_keys: Dict[Tuple[int, str], Dict[str, Any]] = {}
    seen_anchor_only: Dict[int, Dict[str, Any]] = {}

    sorted_proposals = sorted(
        proposals,
        key=lambda row: (
            row.get("page") is None,
            int(row.get("page") or 0),
            -(int(row.get("source_priority") or 0)),
            normalize_heading_key(row.get("title")),
        ),
    )

    for proposal in sorted_proposals:
        title = normalize_heading_display(proposal.get("title"))
        source = str(proposal.get("source") or "")
        key = normalize_heading_key(title)
        repeated = max(int(repeated_pages.get(key, 0)), int(repeated_pages_loose.get(heading_key_without_numbers(title), 0)))
        rejected, reason = is_probable_noise_heading(
            title,
            source=source,
            repeated_pages=repeated,
            repeated_page_threshold=int(options.repeated_heading_page_threshold),
            doc_title_key=doc_title_key,
        )
        anchored = anchor_proposal(proposal, block_index)
        row = dict(anchored)
        row["title"] = title
        row["normalized_title_key"] = key
        row["repeated_pages"] = repeated
        row["accepted"] = False
        row["rejection_reason"] = reason if rejected else ""
        if anchored.get("anchor_abs_block_index") is None:
            row["rejection_reason"] = row["rejection_reason"] or "unanchored"
            rejected = True
        anchored_page = int(anchored.get("anchor_page") or proposal.get("page") or 0)
        titlecase_hits = sum(1 for part in title.split() if part[:1].isupper())
        if (
            not rejected
            and anchored_page == 1
            and not source.startswith("outline")
            and not has_heading_numbering(title)
            and classify_section_type(title) == "body_other"
            and 1 <= count_words(title) <= 10
            and titlecase_hits >= max(1, count_words(title) - 1)
        ):
            row["rejection_reason"] = "probable_author_or_front_matter_line"
            rejected = True
        if not rejected and anchored_page <= 2 and not source.startswith("outline") and title_similarity(title, doc_title) >= 0.85:
            row["rejection_reason"] = "document_title_repeat"
            rejected = True
        if not rejected:
            anchor_abs = int(anchored.get("anchor_abs_block_index") or -1)
            dedupe_key = (anchor_abs, key)
            existing = seen_keys.get(dedupe_key)
            existing_at_anchor = seen_anchor_only.get(anchor_abs)
            if existing is not None:
                if int(anchored.get("source_priority") or 0) > int(existing.get("source_priority") or 0):
                    existing.setdefault("merged_sources", []).append(existing.get("source"))
                    row["merged_sources"] = list(dict.fromkeys((existing.get("merged_sources") or []) + [existing.get("source")]))
                    row["accepted"] = True
                    seen_keys[dedupe_key] = row
                    accepted = [item for item in accepted if item is not existing]
                    accepted.append(row)
                else:
                    existing.setdefault("merged_sources", []).append(source)
                    row["rejection_reason"] = "duplicate_of_higher_priority_heading"
                    rejected = True
            elif existing_at_anchor is not None and (
                str(anchored.get("anchor_method") or "") == "page_start_fallback"
                or str(existing_at_anchor.get("anchor_method") or "") == "page_start_fallback"
            ):
                if int(anchored.get("source_priority") or 0) > int(existing_at_anchor.get("source_priority") or 0):
                    existing_at_anchor.setdefault("merged_sources", []).append(existing_at_anchor.get("source"))
                    row["merged_sources"] = list(dict.fromkeys((existing_at_anchor.get("merged_sources") or []) + [existing_at_anchor.get("source")]))
                    row["accepted"] = True
                    accepted = [item for item in accepted if item is not existing_at_anchor]
                    accepted.append(row)
                    seen_anchor_only[anchor_abs] = row
                    seen_keys[dedupe_key] = row
                else:
                    existing_at_anchor.setdefault("merged_sources", []).append(source)
                    row["rejection_reason"] = "duplicate_anchor_fallback_collision"
                    rejected = True
            else:
                row["accepted"] = True
                row["merged_sources"] = []
                seen_keys[dedupe_key] = row
                seen_anchor_only[anchor_abs] = row
                accepted.append(row)
        proposal_rows.append(row)

    accepted = sorted(
        accepted,
        key=lambda row: (
            int(row.get("anchor_abs_block_index") or 0),
            int(row.get("level_hint") or infer_heading_level(row.get("title"), row.get("source"))),
            -int(row.get("source_priority") or 0),
        ),
    )
    return {"proposal_rows": proposal_rows, "accepted_headings": accepted, "doc_title": doc_title}


def build_section_tree(accepted_headings: List[Dict[str, Any]], *, block_index: Dict[str, Any], metadata: Dict[str, Any], options: PhaseCOptions, doc_id: str, stable_hash_fn: Any) -> List[Dict[str, Any]]:
    ordered_blocks = block_index["ordered_blocks"]
    if not ordered_blocks:
        return []

    headings = list(accepted_headings)
    if not headings and bool(options.synthesize_document_body):
        headings = [
            {
                "proposal_id": "synthetic_document_body",
                "source": "synthetic",
                "title": "Document Body",
                "page": int(ordered_blocks[0].get("page") or 1),
                "level_hint": 1,
                "source_priority": 10,
                "anchor_page": int(ordered_blocks[0].get("page") or 1),
                "anchor_block_index": int(ordered_blocks[0].get("block_index") or 0),
                "anchor_abs_block_index": int(ordered_blocks[0].get("abs_block_index") or 0),
                "anchor_method": "synthetic",
                "anchor_confidence": 1.0,
                "accepted": True,
                "merged_sources": [],
            }
        ]

    headings = sorted(headings, key=lambda row: (int(row.get("anchor_abs_block_index") or 0), -int(row.get("source_priority") or 0)))

    if bool(options.synthesize_front_matter) and headings and int(headings[0].get("anchor_abs_block_index") or 0) > 0:
        first = ordered_blocks[0]
        headings = [
            {
                "proposal_id": "synthetic_front_matter",
                "source": "synthetic",
                "title": "Front Matter",
                "page": int(first.get("page") or 1),
                "level_hint": 1,
                "source_priority": 10,
                "anchor_page": int(first.get("page") or 1),
                "anchor_block_index": int(first.get("block_index") or 0),
                "anchor_abs_block_index": int(first.get("abs_block_index") or 0),
                "anchor_method": "synthetic",
                "anchor_confidence": 1.0,
                "accepted": True,
                "merged_sources": [],
            }
        ] + headings

    deduped: List[Dict[str, Any]] = []
    seen_anchor_titles = set()
    for row in headings:
        key = (int(row.get("anchor_abs_block_index") or 0), normalize_heading_key(row.get("title")))
        if key in seen_anchor_titles:
            continue
        seen_anchor_titles.add(key)
        deduped.append(row)
    headings = deduped

    sections: List[Dict[str, Any]] = []
    for idx, heading in enumerate(headings):
        start_idx = int(heading.get("anchor_abs_block_index") or 0)
        end_idx = int(headings[idx + 1].get("anchor_abs_block_index") or len(ordered_blocks)) - 1 if idx + 1 < len(headings) else len(ordered_blocks) - 1
        start_idx = max(0, min(start_idx, len(ordered_blocks) - 1))
        end_idx = max(start_idx, min(end_idx, len(ordered_blocks) - 1))
        span_blocks = ordered_blocks[start_idx : end_idx + 1]
        block_texts = [clean_text(row.get("text")) for row in span_blocks if clean_text(row.get("text"))]
        section_text = "\n\n".join(block_texts).strip()
        level = infer_heading_level(heading.get("title"), str(heading.get("source") or ""), heading.get("level_hint"))
        section_id = stable_hash_fn(doc_id, f"section::{idx}::{heading.get('title')}::{start_idx}::{end_idx}", length=16)
        sections.append(
            {
                "doc_id": doc_id,
                "section_id": section_id,
                "parent_section_id": None,
                "level": level,
                "title": normalize_heading_display(heading.get("title")),
                "title_path": [],
                "section_type": classify_section_type(heading.get("title")),
                "page_start": int(span_blocks[0].get("page") or 1),
                "page_end": int(span_blocks[-1].get("page") or span_blocks[0].get("page") or 1),
                "char_len": len(section_text),
                "word_count": count_words(section_text),
                "text": section_text,
                "contextualized_text": "",
                "parser_sources": list(dict.fromkeys([str(heading.get("source") or "")] + list(heading.get("merged_sources") or []))),
                "quality_flags": [
                    flag
                    for flag in [
                        "synthetic" if str(heading.get("source") or "") == "synthetic" else "",
                        "fallback_anchor" if str(heading.get("anchor_method") or "") == "page_start_fallback" else "",
                        "tiny_section" if len(section_text) < int(options.min_section_chars) or count_words(section_text) < int(options.min_section_words) else "",
                    ]
                    if flag
                ],
                "heading_anchor": {
                    "page": heading.get("anchor_page"),
                    "block_index": heading.get("anchor_block_index"),
                    "abs_block_index": heading.get("anchor_abs_block_index"),
                    "method": heading.get("anchor_method"),
                    "confidence": heading.get("anchor_confidence"),
                },
                "span": {
                    "start_abs_block_index": start_idx,
                    "end_abs_block_index": end_idx,
                    "block_count": len(span_blocks),
                },
                "block_rows": span_blocks,
            }
        )

    stack: List[Dict[str, Any]] = []
    for section in sections:
        while stack and int(stack[-1]["level"]) >= int(section["level"]):
            stack.pop()
        if stack:
            section["parent_section_id"] = stack[-1]["section_id"]
            section["title_path"] = list(stack[-1]["title_path"]) + [section["title"]]
        else:
            section["title_path"] = [section["title"]]
        stack.append(section)

    doc_title = normalize_heading_display((metadata.get("fitz") or {}).get("metadata", {}).get("title") or metadata.get("label") or metadata.get("file_name"))
    for section in sections:
        path_text = " > ".join(section.get("title_path") or [section.get("title")])
        section["contextualized_text"] = f"Document Title: {doc_title}\nSection Path: {path_text}\n\n{section.get('text') or ''}".strip()

    return sections


def chunk_words(text: str, *, max_words: int) -> List[str]:
    words = re.findall(r"\S+", str(text or ""))
    if not words:
        return []
    chunks: List[str] = []
    for start in range(0, len(words), max_words):
        chunks.append(" ".join(words[start : start + max_words]))
    return chunks


def build_passages(sections: List[Dict[str, Any]], *, metadata: Dict[str, Any], options: PhaseCOptions, stable_hash_fn: Any) -> List[Dict[str, Any]]:
    passages: List[Dict[str, Any]] = []
    doc_title = normalize_heading_display((metadata.get("fitz") or {}).get("metadata", {}).get("title") or metadata.get("label") or metadata.get("file_name"))
    for section in sections:
        current_texts: List[str] = []
        current_pages: List[int] = []
        section_passages: List[Dict[str, Any]] = []
        passage_index = 0

        def flush_current() -> None:
            nonlocal passage_index, current_texts, current_pages, section_passages
            joined = "\n\n".join([part for part in current_texts if part]).strip()
            if not joined:
                current_texts = []
                current_pages = []
                return
            word_count_val = count_words(joined)
            path_text = " > ".join(section.get("title_path") or [section.get("title")])
            if word_count_val > int(options.passage_max_words) * 1.35:
                for sub_idx, part in enumerate(chunk_words(joined, max_words=int(options.passage_max_words))):
                    sub_words = count_words(part)
                    passage_id = stable_hash_fn(section["doc_id"], section["section_id"], f"passage::{passage_index}::{sub_idx}::{part[:80]}", length=16)
                    section_passages.append(
                        {
                            "doc_id": section["doc_id"],
                            "section_id": section["section_id"],
                            "passage_id": passage_id,
                            "passage_index": passage_index,
                            "text": part,
                            "contextualized_text": f"Document Title: {doc_title}\nSection Path: {path_text}\n\n{part}".strip(),
                            "page_span": {
                                "page_start": min(current_pages) if current_pages else section.get("page_start"),
                                "page_end": max(current_pages) if current_pages else section.get("page_end"),
                            },
                            "token_len": max(1, round(sub_words * 1.3)),
                            "word_count": sub_words,
                        }
                    )
                    passage_index += 1
            else:
                passage_id = stable_hash_fn(section["doc_id"], section["section_id"], f"passage::{passage_index}::{joined[:80]}", length=16)
                section_passages.append(
                    {
                        "doc_id": section["doc_id"],
                        "section_id": section["section_id"],
                        "passage_id": passage_id,
                        "passage_index": passage_index,
                        "text": joined,
                        "contextualized_text": f"Document Title: {doc_title}\nSection Path: {path_text}\n\n{joined}".strip(),
                        "page_span": {
                            "page_start": min(current_pages) if current_pages else section.get("page_start"),
                            "page_end": max(current_pages) if current_pages else section.get("page_end"),
                        },
                        "token_len": max(1, round(word_count_val * 1.3)),
                        "word_count": word_count_val,
                    }
                )
                passage_index += 1
            current_texts = []
            current_pages = []

        for row in section.get("block_rows") or []:
            text = clean_text(row.get("text"))
            if not text:
                continue
            words = count_words(text)
            if words >= int(options.passage_max_words) * 1.35:
                flush_current()
                current_texts = [text]
                current_pages = [int(row.get("page") or section.get("page_start") or 1)]
                flush_current()
                continue
            tentative = "\n\n".join(current_texts + [text]) if current_texts else text
            if current_texts and count_words(tentative) > int(options.passage_target_words):
                flush_current()
            current_texts.append(text)
            current_pages.append(int(row.get("page") or section.get("page_start") or 1))
        flush_current()

        if not section_passages and section.get("text"):
            section_text = section.get("text") or ""
            path_text = " > ".join(section.get("title_path") or [section.get("title")])
            passage_id = stable_hash_fn(section["doc_id"], section["section_id"], f"passage::fallback::{section_text[:80]}", length=16)
            section_passages.append(
                {
                    "doc_id": section["doc_id"],
                    "section_id": section["section_id"],
                    "passage_id": passage_id,
                    "passage_index": 0,
                    "text": section_text,
                    "contextualized_text": f"Document Title: {doc_title}\nSection Path: {path_text}\n\n{section_text}".strip(),
                    "page_span": {"page_start": section.get("page_start"), "page_end": section.get("page_end")},
                    "token_len": max(1, round(count_words(section_text) * 1.3)),
                    "word_count": count_words(section_text),
                }
            )
        passages.extend(section_passages)
    return passages


def build_document_record(doc_id: str, metadata: Dict[str, Any], sections: List[Dict[str, Any]], accepted_headings: List[Dict[str, Any]], loaded_bundle: Dict[str, Any]) -> Dict[str, Any]:
    title = normalize_heading_display((metadata.get("fitz") or {}).get("metadata", {}).get("title") or metadata.get("label") or metadata.get("file_name"))
    sample_text = "\n".join(str(row.get("text") or "") for row in (loaded_bundle.get("pages") or [])[:2])
    page_count = int(metadata.get("page_count") or 0)
    return {
        "doc_id": doc_id,
        "source_path": metadata.get("source_path"),
        "sha256": metadata.get("sha256"),
        "title": title,
        "page_count": page_count,
        "language_guess": infer_language_guess(title + " " + sample_text),
        "doc_type_guess": infer_doc_type_guess(metadata),
        "has_outline": bool((metadata.get("outline_counts") or {}).get("fitz") or (metadata.get("outline_counts") or {}).get("pypdf")),
        "section_count": len(sections),
        "accepted_heading_count": len(accepted_headings),
    }


def assess_phase_c(*, summary_rows: List[Dict[str, Any]], section_rows: List[Dict[str, Any]], passage_rows: List[Dict[str, Any]], options: PhaseCOptions) -> Dict[str, Any]:
    failures: List[str] = []
    warnings: List[str] = []
    if not summary_rows:
        failures.append("No documents were processed in Phase C.")
    if not section_rows:
        failures.append("No sections were produced.")
    if not passage_rows:
        failures.append("No passages were produced.")

    doc_ids_with_sections = {row.get("doc_id") for row in section_rows}
    doc_ids_with_passages = {row.get("doc_id") for row in passage_rows}
    section_ids = {row.get("section_id") for row in section_rows}
    for row in summary_rows:
        doc_id = row.get("doc_id")
        if doc_id not in doc_ids_with_sections:
            failures.append(f"{doc_id}: no sections")
        if doc_id not in doc_ids_with_passages:
            failures.append(f"{doc_id}: no passages")
        if float(row.get("section_coverage_pct") or 0.0) < float(options.min_section_coverage_pct_warn):
            warnings.append(f"{doc_id}: low section coverage ({row.get('section_coverage_pct')}%)")
        if int(row.get("section_count") or 0) <= 1 and int(row.get("page_count") or 0) >= 10:
            warnings.append(f"{doc_id}: collapsed to a single section on a multi-page document")
        if int(row.get("page_count") or 0) >= int(options.long_doc_page_threshold) and not bool(row.get("has_references_section")):
            warnings.append(f"{doc_id}: no references section detected in a long document")
        if int(row.get("fallback_anchor_count") or 0) > 0:
            warnings.append(f"{doc_id}: {row.get('fallback_anchor_count')} headings used fallback anchoring")

    orphan_passages = [row.get("passage_id") for row in passage_rows if row.get("section_id") not in section_ids]
    if orphan_passages:
        failures.append(f"Orphan passages detected: {len(orphan_passages)}")

    status = "success"
    quality_band = "high"
    if failures:
        status = "failed"
        quality_band = "insufficient"
    elif warnings:
        status = "success_with_warnings"
        quality_band = "acceptable_with_issues"

    low_coverage_docs = [row for row in summary_rows if float(row.get("section_coverage_pct") or 0.0) < float(options.min_section_coverage_pct_warn)]
    qc_rows = [
        qc_row(
            check="documents_processed",
            status="OK" if summary_rows else "FAIL",
            value=len(summary_rows),
            expected=">= 1",
            why="Phase C needs at least one normalized document.",
            fix="check Phase B outputs and Phase C doc filters",
        ),
        qc_row(
            check="sections_produced",
            status="OK" if section_rows else "FAIL",
            value=len(section_rows),
            expected=">= 1",
            why="Section ranking depends on canonical sections.",
            fix="inspect accepted_headings.jsonl and phase_c_diagnostics.json",
        ),
        qc_row(
            check="passages_produced",
            status="OK" if passage_rows else "FAIL",
            value=len(passage_rows),
            expected=">= 1",
            why="Later retrieval and evidence display depend on passages.",
            fix="inspect sections.jsonl and passage chunking options",
        ),
        qc_row(
            check="orphan_passages",
            status="OK" if not orphan_passages else "FAIL",
            value="none" if not orphan_passages else len(orphan_passages),
            expected="none",
            why="Every passage must map back to one canonical section.",
            fix="inspect section_id assignment during passage construction",
        ),
        qc_row(
            check="low_coverage_docs",
            status="OK" if not low_coverage_docs else "WARN",
            value="none" if not low_coverage_docs else ", ".join(row.get("doc_id") for row in low_coverage_docs),
            expected=f">= {options.min_section_coverage_pct_warn}% section text coverage",
            why="Low coverage indicates that headings or section spans are dropping content.",
            fix="inspect section span boundaries and fallback synthesis",
        ),
    ]

    return {
        "status": status,
        "quality_band": quality_band,
        "can_continue_to_next_phase": not failures,
        "failures": failures,
        "warnings": warnings,
        "counts": {
            "document_count": len(summary_rows),
            "section_count": len(section_rows),
            "passage_count": len(passage_rows),
            "orphan_passage_count": len(orphan_passages),
            "warning_count": len(warnings),
            "failure_count": len(failures),
        },
        "qc_rows": qc_rows,
    }


def run_phase_c(run_ctx: Any, options: PhaseCOptions, *, stable_hash_fn=None, log_event_fn=None, run_logger=None) -> Dict[str, Any]:
    options = options.normalized()
    stable_hash_local = stable_hash_fn or stable_hash
    normalized_dir = ensure_dir(Path(run_ctx.artifacts.normalized_dir))
    config_path = normalized_dir / "phase_c_config.json"
    runtime_path = normalized_dir / "phase_c_runtime.json"
    summary_path = normalized_dir / "phase_c_summary.json"
    assessment_path = normalized_dir / "phase_c_assessment.json"
    documents_path = normalized_dir / "documents.jsonl"
    sections_path = normalized_dir / "sections.jsonl"
    passages_path = normalized_dir / "passages.jsonl"
    index_path = normalized_dir / "normalized_document_bundles.jsonl"

    phase_b_assessment_path = Path(run_ctx.artifacts.parser_dir) / "phase_b_assessment.json"
    phase_b_index_path = Path(run_ctx.artifacts.parser_dir) / "parsed_document_bundles.jsonl"
    if not phase_b_index_path.exists():
        raise FileNotFoundError(f"Phase B index not found: {phase_b_index_path}")

    phase_b_assessment = read_json(phase_b_assessment_path) if phase_b_assessment_path.exists() else {}
    phase_b_can_continue = bool(((phase_b_assessment or {}).get("assessment") or {}).get("can_continue_to_next_phase", True))
    if not phase_b_can_continue:
        raise RuntimeError("Phase B assessment does not allow continuation to Phase C.")

    bundle_rows = read_jsonl_rows(phase_b_index_path)
    include_set = set(options.include_doc_ids or [])
    exclude_set = set(options.exclude_doc_ids or [])
    selected_bundles: List[Dict[str, Any]] = []
    for row in bundle_rows:
        doc_id = str(row.get("doc_id") or "")
        if include_set and doc_id not in include_set:
            continue
        if doc_id in exclude_set:
            continue
        selected_bundles.append(row)
    if options.doc_limit is not None:
        selected_bundles = selected_bundles[: int(options.doc_limit)]

    runtime_payload = {
        "generated_at_utc": utc_now_iso(),
        "phase": "phase_c",
        "options": json_safe(asdict(options)),
        "python_runtime": {
            "python_executable": sys.executable,
            "python_version": sys.version.split()[0],
        },
        "phase_b_assessment_path": rel_to_run(Path(run_ctx.run_dir), phase_b_assessment_path),
        "phase_b_index_path": rel_to_run(Path(run_ctx.run_dir), phase_b_index_path),
        "selected_doc_count": len(selected_bundles),
    }
    write_json_atomic(runtime_path, runtime_payload)
    write_json_atomic(config_path, {"generated_at_utc": utc_now_iso(), "phase": "phase_c", "options": json_safe(asdict(options))})

    if run_logger is not None:
        run_logger.info(
            "Phase C start | selected_docs=%s | prefer_outline=%s | use_docling=%s | use_grobid=%s | use_heuristics=%s",
            len(selected_bundles),
            options.prefer_outline,
            options.use_docling,
            options.use_grobid,
            options.use_heuristic_headings,
        )
    if log_event_fn is not None:
        log_event_fn(run_ctx, stage="phase_c", event="phase_started", selected_doc_count=len(selected_bundles), options=json_safe(asdict(options)))

    document_rows: List[Dict[str, Any]] = []
    section_rows: List[Dict[str, Any]] = []
    passage_rows: List[Dict[str, Any]] = []
    summary_rows: List[Dict[str, Any]] = []
    bundle_index_rows: List[Dict[str, Any]] = []

    for bundle_row in selected_bundles:
        doc_id = str(bundle_row.get("doc_id") or "")
        doc_dir = ensure_dir(normalized_dir / doc_id)
        doc_artifacts = {
            "document_json": doc_dir / "document.json",
            "section_proposals_jsonl": doc_dir / "section_proposals.jsonl",
            "accepted_headings_jsonl": doc_dir / "accepted_headings.jsonl",
            "sections_jsonl": doc_dir / "sections.jsonl",
            "passages_jsonl": doc_dir / "passages.jsonl",
            "diagnostics_json": doc_dir / "phase_c_diagnostics.json",
        }

        use_cache = bool(not options.force_rebuild and all(path.exists() for path in doc_artifacts.values()))
        if use_cache:
            doc_record = read_json(doc_artifacts["document_json"])
            doc_sections = read_jsonl_rows(doc_artifacts["sections_jsonl"])
            doc_passages = read_jsonl_rows(doc_artifacts["passages_jsonl"])
            diagnostics = read_json(doc_artifacts["diagnostics_json"])
            document_rows.append(doc_record)
            section_rows.extend(doc_sections)
            passage_rows.extend(doc_passages)
            summary_rows.append(diagnostics.get("summary_row") or {})
            bundle_index_rows.append(
                {
                    "doc_id": doc_id,
                    "document_json": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["document_json"]),
                    "section_proposals_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["section_proposals_jsonl"]),
                    "accepted_headings_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["accepted_headings_jsonl"]),
                    "sections_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["sections_jsonl"]),
                    "passages_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["passages_jsonl"]),
                    "diagnostics_json": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["diagnostics_json"]),
                    "bundle_status": "cached",
                }
            )
            if run_logger is not None:
                run_logger.info("Phase C doc cached | doc_id=%s | sections=%s | passages=%s", doc_id, len(doc_sections), len(doc_passages))
            continue

        loaded = load_phase_b_bundle(run_ctx, bundle_row)
        metadata = loaded["metadata"] or {}
        block_index = build_block_index(loaded["blocks"] or [])
        proposals: List[Dict[str, Any]] = []
        source_counts: Dict[str, int] = {}
        outline_props: List[Dict[str, Any]] = []
        docling_props: List[Dict[str, Any]] = []
        grobid_props: List[Dict[str, Any]] = []
        if bool(options.prefer_outline):
            outline_props = extract_outline_proposals(metadata)
            proposals.extend(outline_props)
            source_counts["outline"] = len(outline_props)
        if bool(options.use_docling):
            docling_props = extract_docling_proposals(loaded["docling"] or {})
            proposals.extend(docling_props)
            source_counts["docling"] = len(docling_props)
        if bool(options.use_grobid):
            grobid_props = extract_grobid_proposals(loaded.get("grobid_tei_text") or "")
            proposals.extend(grobid_props)
            source_counts["grobid"] = len(grobid_props)
        heuristic_enabled_for_doc = bool(
            options.use_heuristic_headings
            and not outline_props
            and len(docling_props) < 4
            and len(grobid_props) < 4
        )
        if heuristic_enabled_for_doc:
            heuristic_props = extract_heuristic_proposals(block_index["ordered_blocks"], options)
            proposals.extend(heuristic_props)
            source_counts["heuristic"] = len(heuristic_props)
        else:
            source_counts["heuristic"] = 0

        filtered = filter_and_anchor_proposals(proposals, block_index=block_index, metadata=metadata, options=options)
        accepted_headings = filtered["accepted_headings"]
        sections = build_section_tree(
            accepted_headings,
            block_index=block_index,
            metadata=metadata,
            options=options,
            doc_id=doc_id,
            stable_hash_fn=stable_hash_local,
        )
        doc_record = build_document_record(doc_id, metadata, sections, accepted_headings, loaded)
        passages = build_passages(sections, metadata=metadata, options=options, stable_hash_fn=stable_hash_local)

        section_export_rows: List[Dict[str, Any]] = []
        for row in sections:
            export_row = {k: v for k, v in row.items() if k != "block_rows"}
            section_export_rows.append(export_row)

        covered_abs_indices = {
            idx
            for row in sections
            for idx in range(int((row.get("span") or {}).get("start_abs_block_index") or 0), int((row.get("span") or {}).get("end_abs_block_index") or -1) + 1)
        }
        coverage_chars = sum(
            int(block.get("char_len") or len(block.get("text") or ""))
            for block in block_index["ordered_blocks"]
            if int(block.get("abs_block_index") or -1) in covered_abs_indices
        )
        total_chars = int(block_index.get("total_block_chars") or 0)
        coverage_pct = round((coverage_chars / total_chars) * 100.0, 2) if total_chars else 0.0
        fallback_anchor_count = sum(1 for row in accepted_headings if str(row.get("anchor_method") or "") == "page_start_fallback")
        section_types = [row.get("section_type") for row in sections]
        summary_row = {
            "doc_id": doc_id,
            "file_name": metadata.get("file_name"),
            "page_count": metadata.get("page_count"),
            "outline_count": (metadata.get("outline_counts") or {}).get("fitz") or (metadata.get("outline_counts") or {}).get("pypdf") or 0,
            "proposal_count": len(filtered["proposal_rows"]),
            "accepted_heading_count": len(accepted_headings),
            "section_count": len(section_export_rows),
            "passage_count": len(passages),
            "section_coverage_pct": coverage_pct,
            "fallback_anchor_count": fallback_anchor_count,
            "docling_status": (loaded.get("docling") or {}).get("status"),
            "grobid_status": (loaded.get("grobid_summary") or {}).get("status") or (loaded.get("grobid_summary") or {}).get("service_status") or "no_data",
            "strategy": "outline_first" if source_counts.get("outline") else ("docling_first" if source_counts.get("docling") else "heuristic_only"),
            "heading_sources": ", ".join(sorted({str(row.get("source") or "") for row in accepted_headings})) or "none",
            "has_references_section": bool(any(item == "references" for item in section_types)),
            "has_appendix_section": bool(any(item == "appendix" for item in section_types)),
            "collapsed_to_single_section": bool(len(section_export_rows) <= 1),
        }
        diagnostics = {
            "generated_at_utc": utc_now_iso(),
            "phase": "phase_c",
            "doc_id": doc_id,
            "source_counts": source_counts,
            "proposal_count": len(filtered["proposal_rows"]),
            "accepted_heading_count": len(accepted_headings),
            "section_count": len(section_export_rows),
            "passage_count": len(passages),
            "section_coverage_pct": coverage_pct,
            "summary_row": summary_row,
            "notes": {
                "doc_title": filtered.get("doc_title"),
                "docling_status": (loaded.get("docling") or {}).get("status"),
                "grobid_status": (loaded.get("grobid_summary") or {}).get("status") or (loaded.get("grobid_summary") or {}).get("service_status") or "no_data",
                "heuristics_enabled_for_doc": heuristic_enabled_for_doc,
            },
        }

        write_json_atomic(doc_artifacts["document_json"], doc_record)
        write_jsonl_rows(doc_artifacts["section_proposals_jsonl"], filtered["proposal_rows"])
        write_jsonl_rows(doc_artifacts["accepted_headings_jsonl"], accepted_headings)
        write_jsonl_rows(doc_artifacts["sections_jsonl"], section_export_rows)
        write_jsonl_rows(doc_artifacts["passages_jsonl"], passages)
        write_json_atomic(doc_artifacts["diagnostics_json"], diagnostics)

        document_rows.append(doc_record)
        section_rows.extend(section_export_rows)
        passage_rows.extend(passages)
        summary_rows.append(summary_row)
        bundle_index_rows.append(
            {
                "doc_id": doc_id,
                "document_json": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["document_json"]),
                "section_proposals_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["section_proposals_jsonl"]),
                "accepted_headings_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["accepted_headings_jsonl"]),
                "sections_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["sections_jsonl"]),
                "passages_jsonl": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["passages_jsonl"]),
                "diagnostics_json": rel_to_run(Path(run_ctx.run_dir), doc_artifacts["diagnostics_json"]),
                "bundle_status": "ok",
            }
        )
        if run_logger is not None:
            run_logger.info(
                "Phase C doc built | doc_id=%s | proposals=%s | accepted_headings=%s | sections=%s | passages=%s | coverage_pct=%s",
                doc_id,
                len(filtered["proposal_rows"]),
                len(accepted_headings),
                len(section_export_rows),
                len(passages),
                coverage_pct,
            )
        if log_event_fn is not None:
            log_event_fn(
                run_ctx,
                stage="phase_c",
                event="doc_normalized",
                doc_id=doc_id,
                proposal_count=len(filtered["proposal_rows"]),
                accepted_heading_count=len(accepted_headings),
                section_count=len(section_export_rows),
                passage_count=len(passages),
                section_coverage_pct=coverage_pct,
            )

    write_jsonl_rows(documents_path, document_rows)
    write_jsonl_rows(sections_path, section_rows)
    write_jsonl_rows(passages_path, passage_rows)
    write_jsonl_rows(index_path, bundle_index_rows)

    assessment = assess_phase_c(summary_rows=summary_rows, section_rows=section_rows, passage_rows=passage_rows, options=options)
    assessment_payload = {
        "generated_at_utc": utc_now_iso(),
        "run_id": run_ctx.run_id,
        "phase": "phase_c",
        "assessment": {k: v for k, v in assessment.items() if k != "qc_rows"},
        "qc_rows": assessment["qc_rows"],
        "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
        "summary_path": rel_to_run(Path(run_ctx.run_dir), summary_path),
        "index_path": rel_to_run(Path(run_ctx.run_dir), index_path),
    }
    summary_payload = {
        "generated_at_utc": utc_now_iso(),
        "run_id": run_ctx.run_id,
        "phase": "phase_c",
        "options": json_safe(asdict(options)),
        "runtime_path": rel_to_run(Path(run_ctx.run_dir), runtime_path),
        "assessment": assessment_payload["assessment"],
        "qc_rows": assessment["qc_rows"],
        "documents": summary_rows,
        "artifacts": bundle_index_rows,
    }
    write_json_atomic(assessment_path, assessment_payload)
    write_json_atomic(summary_path, summary_payload)

    if run_logger is not None:
        run_logger.info(
            "Phase C finished | status=%s | quality_band=%s | documents=%s | sections=%s | passages=%s",
            assessment_payload["assessment"].get("status"),
            assessment_payload["assessment"].get("quality_band"),
            len(summary_rows),
            len(section_rows),
            len(passage_rows),
        )
    if log_event_fn is not None:
        log_event_fn(
            run_ctx,
            stage="phase_c",
            event="phase_finished",
            status=assessment_payload["assessment"].get("status"),
            quality_band=assessment_payload["assessment"].get("quality_band"),
            document_count=len(summary_rows),
            section_count=len(section_rows),
            passage_count=len(passage_rows),
        )

    return {
        "config_path": config_path,
        "runtime_path": runtime_path,
        "summary_path": summary_path,
        "assessment_path": assessment_path,
        "documents_path": documents_path,
        "sections_path": sections_path,
        "passages_path": passages_path,
        "index_path": index_path,
        "summary_rows": summary_rows,
        "bundle_rows": bundle_index_rows,
        "document_rows": document_rows,
        "section_rows": section_rows,
        "passage_rows": passage_rows,
        "assessment": assessment_payload["assessment"],
        "qc_rows": assessment["qc_rows"],
        "selected_count": len(selected_bundles),
        "metrics_update": {
            "status": assessment_payload["assessment"].get("status"),
            "quality_band": assessment_payload["assessment"].get("quality_band"),
            "document_count": len(summary_rows),
            "section_count": len(section_rows),
            "passage_count": len(passage_rows),
            "warning_count": assessment_payload["assessment"].get("counts", {}).get("warning_count"),
            "failure_count": assessment_payload["assessment"].get("counts", {}).get("failure_count"),
            "phase_c_summary_path": rel_to_run(Path(run_ctx.run_dir), summary_path),
            "phase_c_assessment_path": rel_to_run(Path(run_ctx.run_dir), assessment_path),
        },
    }


In [ ]:
# Phase C.1 - Run normalization, build canonical sections/passages, and print QC summary

phase_c_options = PhaseCOptions(
    force_rebuild=True,
    doc_limit=None,
    include_doc_ids=[],
    exclude_doc_ids=[],
    prefer_outline=True,
    use_docling=True,
    use_grobid=True,
    use_heuristic_headings=True,
    heuristic_heading_min_words=1,
    heuristic_heading_max_words=18,
    heuristic_heading_max_chars=160,
    repeated_heading_page_threshold=3,
    min_section_chars=120,
    min_section_words=20,
    min_section_coverage_pct_warn=70.0,
    long_doc_page_threshold=40,
    passage_target_words=180,
    passage_max_words=260,
    passage_min_words=70,
    synthesize_front_matter=True,
    synthesize_document_body=True,
)

phase_c_logger = setup_run_logger(RUN_CONTEXT)

with stage_timer(RUN_CONTEXT, "phase_c"):
    phase_c_result = run_phase_c(
        RUN_CONTEXT,
        phase_c_options,
        stable_hash_fn=stable_hash,
        log_event_fn=log_event,
        run_logger=phase_c_logger,
    )
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_c", {}).update(phase_c_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_c_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))

PHASE_C_RESULT = phase_c_result
PHASE_C_SUMMARY = phase_c_result["summary_rows"]
PHASE_C_BUNDLES = phase_c_result["bundle_rows"]
PHASE_C_DOCUMENTS = phase_c_result["document_rows"]
PHASE_C_SECTIONS = phase_c_result["section_rows"]
PHASE_C_PASSAGES = phase_c_result["passage_rows"]

outline_strategy_docs = sum(1 for row in PHASE_C_SUMMARY if str(row.get("strategy") or "") == "outline_first")
docling_strategy_docs = sum(1 for row in PHASE_C_SUMMARY if str(row.get("strategy") or "") == "docling_first")
heuristic_strategy_docs = sum(1 for row in PHASE_C_SUMMARY if str(row.get("strategy") or "") == "heuristic_only")
fallback_anchor_docs = sum(1 for row in PHASE_C_SUMMARY if int(row.get("fallback_anchor_count") or 0) > 0)

print_section("Phase C - Normalization Capabilities")
print_kv(
    {
        "selected_documents": phase_c_result["selected_count"],
        "outline_first_docs": outline_strategy_docs,
        "docling_first_docs": docling_strategy_docs,
        "heuristic_only_docs": heuristic_strategy_docs,
        "documents_with_fallback_anchors": fallback_anchor_docs,
        "documents_jsonl": phase_c_rel(phase_c_result["documents_path"]),
        "sections_jsonl": phase_c_rel(phase_c_result["sections_path"]),
        "passages_jsonl": phase_c_rel(phase_c_result["passages_path"]),
    }
)

print_section("Phase C - What Happened")
print_kv(
    {
        "phase_c_config_json": phase_c_rel(phase_c_result["config_path"]),
        "phase_c_runtime_json": phase_c_rel(phase_c_result["runtime_path"]),
        "phase_c_summary_json": phase_c_rel(phase_c_result["summary_path"]),
        "phase_c_assessment_json": phase_c_rel(phase_c_result["assessment_path"]),
        "normalized_document_bundles_jsonl": phase_c_rel(phase_c_result["index_path"]),
        "documents_processed": len(PHASE_C_SUMMARY),
        "documents_written": len(PHASE_C_DOCUMENTS),
        "sections_written": len(PHASE_C_SECTIONS),
        "passages_written": len(PHASE_C_PASSAGES),
        "phase_status": phase_c_result["assessment"].get("status"),
        "quality_band": phase_c_result["assessment"].get("quality_band"),
    }
)

print_section("Phase C - Document Summary")
print_table(
    PHASE_C_SUMMARY,
    columns=[
        "doc_id",
        "file_name",
        "page_count",
        "strategy",
        "accepted_heading_count",
        "section_count",
        "passage_count",
        "section_coverage_pct",
        "fallback_anchor_count",
    ],
    max_rows=20,
    max_col_width=44,
)

print_section("Phase C - Artifact Preview")
print_table(
    PHASE_C_BUNDLES,
    columns=[
        "doc_id",
        "document_json",
        "section_proposals_jsonl",
        "accepted_headings_jsonl",
        "sections_jsonl",
        "passages_jsonl",
        "bundle_status",
    ],
    max_rows=20,
    max_col_width=44,
)

print_section("Phase C - Assessment")
print_kv(
    {
        "status": phase_c_result["assessment"].get("status"),
        "quality_band": phase_c_result["assessment"].get("quality_band"),
        "can_continue": phase_c_result["assessment"].get("can_continue_to_next_phase"),
        "warning_count": len(phase_c_result["assessment"].get("warnings") or []),
        "failure_count": len(phase_c_result["assessment"].get("failures") or []),
        "document_count": phase_c_result["assessment"].get("counts", {}).get("document_count"),
        "section_count": phase_c_result["assessment"].get("counts", {}).get("section_count"),
        "passage_count": phase_c_result["assessment"].get("counts", {}).get("passage_count"),
    }
)

print_section("Phase C - QC")
print_table(
    phase_c_result["qc_rows"],
    columns=["check", "status", "value", "expected", "why", "fix"],
    max_rows=20,
    max_col_width=46,
)


In [ ]:
# Phase C.2 - Print section inspection preview so section selection can be judged quickly

def build_phase_c_preview_rows(summary_rows, section_rows, *, per_doc: int = 6, preview_chars: int = 180) -> List[Dict[str, Any]]:
    sections_by_doc: Dict[str, List[Dict[str, Any]]] = {}
    for row in section_rows or []:
        sections_by_doc.setdefault(str(row.get("doc_id") or ""), []).append(row)

    priority_types = ["introduction", "background", "methods", "results", "discussion", "conclusion", "references", "appendix"]
    preview_rows: List[Dict[str, Any]] = []

    for summary in summary_rows or []:
        doc_id = str(summary.get("doc_id") or "")
        file_name = str(summary.get("file_name") or "")
        doc_sections = sorted(
            sections_by_doc.get(doc_id, []),
            key=lambda row: (
                int(row.get("page_start") or 0),
                int((row.get("span") or {}).get("start_abs_block_index") or 0),
            ),
        )
        selected: List[Dict[str, Any]] = []
        selected_ids = set()

        def add_row(row: Optional[Dict[str, Any]]) -> None:
            if not row:
                return
            section_id = str(row.get("section_id") or "")
            if not section_id or section_id in selected_ids:
                return
            selected.append(row)
            selected_ids.add(section_id)

        for row in doc_sections[:3]:
            add_row(row)

        for section_type in priority_types:
            match = next((row for row in doc_sections if str(row.get("section_type") or "") == section_type), None)
            add_row(match)
            if len(selected) >= int(per_doc):
                break

        for row in doc_sections:
            if len(selected) >= int(per_doc):
                break
            add_row(row)

        for row in selected[: int(per_doc)]:
            preview_rows.append(
                {
                    "doc_id": doc_id,
                    "file_name": file_name,
                    "title": row.get("title"),
                    "section_type": row.get("section_type"),
                    "pages": f"{row.get('page_start')}-{row.get('page_end')}",
                    "parser_sources": ", ".join(row.get("parser_sources") or []),
                    "text_preview": _truncate(clean_text(row.get("text")), max_len=preview_chars),
                }
            )

    return preview_rows


PHASE_C_PREVIEW_ROWS = build_phase_c_preview_rows(PHASE_C_SUMMARY, PHASE_C_SECTIONS, per_doc=6, preview_chars=180)

print_section("Phase C - Section Inspection Preview")
print_table(
    PHASE_C_PREVIEW_ROWS,
    columns=["doc_id", "title", "section_type", "pages", "parser_sources", "text_preview"],
    max_rows=40,
    max_col_width=52,
)


---

## Phase D - Query planner and retrieval views


In [ ]:
# Phase D.0 - Query planner and derived retrieval-view helpers

import json
import re
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

PHASE_D_OPTIONAL_IMPORT_ERRORS: Dict[str, str] = {}

try:
    from openai import OpenAI
except Exception as e:
    OpenAI = None
    PHASE_D_OPTIONAL_IMPORT_ERRORS["openai"] = f"{type(e).__name__}: {e}"

try:
    from pydantic import BaseModel, Field, ValidationError
except Exception as e:
    BaseModel = None
    Field = None
    ValidationError = Exception
    PHASE_D_OPTIONAL_IMPORT_ERRORS["pydantic"] = f"{type(e).__name__}: {e}"


PHASE_D_ALLOWED_SECTION_TYPES = [
    "front_matter",
    "abstract",
    "introduction",
    "background",
    "related_work",
    "methods",
    "results",
    "discussion",
    "conclusion",
    "body_other",
    "references",
    "appendix",
    "acknowledgements",
    "table_of_contents",
    "index",
]


@dataclass
class PhaseDOptions:
    force_rebuild: bool = False
    use_openai_planner: bool = True
    allow_heuristic_fallback: bool = True
    openai_model: str = "gpt-5-mini"
    reasoning_effort: str = "medium"
    temperature: float = 0.0
    max_completion_tokens: int = 1800
    must_term_limit: int = 12
    should_term_limit: int = 18
    exclusion_limit: int = 8
    subpoint_limit: int = 6
    drift_risk_limit: int = 8

    def normalized(self) -> "PhaseDOptions":
        return PhaseDOptions(
            force_rebuild=bool(self.force_rebuild),
            use_openai_planner=bool(self.use_openai_planner),
            allow_heuristic_fallback=bool(self.allow_heuristic_fallback),
            openai_model=str(self.openai_model or "gpt-5-mini").strip() or "gpt-5-mini",
            reasoning_effort=str(self.reasoning_effort or "medium").strip() or "medium",
            temperature=float(self.temperature),
            max_completion_tokens=int(self.max_completion_tokens),
            must_term_limit=int(self.must_term_limit),
            should_term_limit=int(self.should_term_limit),
            exclusion_limit=int(self.exclusion_limit),
            subpoint_limit=int(self.subpoint_limit),
            drift_risk_limit=int(self.drift_risk_limit),
        )


if BaseModel is not None:
    class QuerySubpointModel(BaseModel):
        subpoint_id: str = Field(min_length=1)
        label: str = Field(min_length=1)
        summary: str = Field(min_length=1)
        must_terms: List[str] = Field(default_factory=list)
        should_terms: List[str] = Field(default_factory=list)
        preferred_section_types: List[str] = Field(default_factory=list)


    class QueryPlanModel(BaseModel):
        query_id: str = Field(min_length=1)
        chapter_title: str = Field(min_length=1)
        chapter_summary: str = Field(min_length=1)
        must_terms: List[str] = Field(default_factory=list)
        should_terms: List[str] = Field(default_factory=list)
        exclusions: List[str] = Field(default_factory=list)
        subpoints: List[QuerySubpointModel] = Field(default_factory=list)
        language_hints: List[str] = Field(default_factory=list)
        preferred_section_types: List[str] = Field(default_factory=list)
        penalized_section_types: List[str] = Field(default_factory=list)
        drift_risks: List[str] = Field(default_factory=list)
else:
    QuerySubpointModel = None
    QueryPlanModel = None


def phase_d_capabilities() -> Dict[str, Any]:
    return {
        "python_executable": sys.executable,
        "python_version": sys.version.split()[0],
        "openai_available": bool(OpenAI is not None),
        "pydantic_available": bool(BaseModel is not None),
        "openai_api_key_present": bool(OPENAI_API_KEY),
        "optional_import_errors": dict(PHASE_D_OPTIONAL_IMPORT_ERRORS),
    }


OPENAI_API_PRICING_SOURCE_URL = "https://openai.com/api/pricing/"
OPENAI_API_PRICING_VERIFIED_DATE = "2026-03-14"
OPENAI_TEXT_MODEL_PRICING_USD_PER_1M = {
    "gpt-5": {"input": 1.25, "cached_input": 0.125, "output": 10.0},
    "gpt-5-mini": {"input": 0.25, "cached_input": 0.025, "output": 2.0},
    "gpt-5-nano": {"input": 0.05, "cached_input": 0.005, "output": 0.4},
}


def resolve_openai_text_model_pricing(model_name: str) -> Dict[str, Any]:
    model_key = str(model_name or "").strip()
    for pricing_model in sorted(OPENAI_TEXT_MODEL_PRICING_USD_PER_1M.keys(), key=len, reverse=True):
        if model_key == pricing_model or model_key.startswith(pricing_model + "-"):
            return {
                "pricing_found": True,
                "pricing_model": pricing_model,
                "model_name": model_key,
                "pricing_source_url": OPENAI_API_PRICING_SOURCE_URL,
                "pricing_verified_date": OPENAI_API_PRICING_VERIFIED_DATE,
                "rates_usd_per_1m_tokens": dict(OPENAI_TEXT_MODEL_PRICING_USD_PER_1M[pricing_model]),
            }
    return {
        "pricing_found": False,
        "pricing_model": None,
        "model_name": model_key or None,
        "pricing_source_url": OPENAI_API_PRICING_SOURCE_URL,
        "pricing_verified_date": OPENAI_API_PRICING_VERIFIED_DATE,
        "rates_usd_per_1m_tokens": None,
    }


def extract_openai_usage_payload(usage: Any) -> Dict[str, Any]:
    prompt_details = getattr(usage, "prompt_tokens_details", None) or getattr(usage, "input_tokens_details", None)
    completion_details = getattr(usage, "completion_tokens_details", None) or getattr(usage, "output_tokens_details", None)
    input_tokens = getattr(usage, "prompt_tokens", None)
    if input_tokens is None:
        input_tokens = getattr(usage, "input_tokens", None)
    output_tokens = getattr(usage, "completion_tokens", None)
    if output_tokens is None:
        output_tokens = getattr(usage, "output_tokens", None)
    total_tokens = getattr(usage, "total_tokens", None)
    cached_input_tokens = getattr(prompt_details, "cached_tokens", None) if prompt_details is not None else None
    reasoning_tokens = getattr(completion_details, "reasoning_tokens", None) if completion_details is not None else None
    accepted_prediction_tokens = getattr(completion_details, "accepted_prediction_tokens", None) if completion_details is not None else None
    rejected_prediction_tokens = getattr(completion_details, "rejected_prediction_tokens", None) if completion_details is not None else None
    audio_input_tokens = getattr(prompt_details, "audio_tokens", None) if prompt_details is not None else None
    audio_output_tokens = getattr(completion_details, "audio_tokens", None) if completion_details is not None else None
    non_cached_input_tokens = None
    if isinstance(input_tokens, int):
        non_cached_input_tokens = int(input_tokens)
        if isinstance(cached_input_tokens, int):
            non_cached_input_tokens = max(int(input_tokens) - int(cached_input_tokens), 0)
    return {
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_input_tokens,
        "non_cached_input_tokens": non_cached_input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "accepted_prediction_tokens": accepted_prediction_tokens,
        "rejected_prediction_tokens": rejected_prediction_tokens,
        "audio_input_tokens": audio_input_tokens,
        "audio_output_tokens": audio_output_tokens,
        "total_tokens": total_tokens,
    }


def estimate_openai_text_cost_usd(model_name: str, usage_payload: Dict[str, Any]) -> Dict[str, Any]:
    pricing_info = resolve_openai_text_model_pricing(model_name)
    usage_payload = dict(usage_payload or {})
    result = {
        **pricing_info,
        "usage": usage_payload,
        "estimated_cost_usd": None,
        "cost_components_usd": {},
    }
    rates = pricing_info.get("rates_usd_per_1m_tokens") or {}
    if not pricing_info.get("pricing_found"):
        return result

    input_tokens = usage_payload.get("input_tokens")
    cached_input_tokens = usage_payload.get("cached_input_tokens")
    output_tokens = usage_payload.get("output_tokens")
    non_cached_input_tokens = usage_payload.get("non_cached_input_tokens")
    if non_cached_input_tokens is None and isinstance(input_tokens, int):
        non_cached_input_tokens = int(input_tokens)
        if isinstance(cached_input_tokens, int):
            non_cached_input_tokens = max(int(input_tokens) - int(cached_input_tokens), 0)

    total_cost = 0.0
    if isinstance(non_cached_input_tokens, int):
        input_cost = (non_cached_input_tokens / 1_000_000.0) * float(rates.get("input") or 0.0)
        result["cost_components_usd"]["input_cost_usd"] = round(input_cost, 8)
        total_cost += input_cost
    if isinstance(cached_input_tokens, int):
        cached_cost = (cached_input_tokens / 1_000_000.0) * float(rates.get("cached_input") or 0.0)
        result["cost_components_usd"]["cached_input_cost_usd"] = round(cached_cost, 8)
        total_cost += cached_cost
    if isinstance(output_tokens, int):
        output_cost = (output_tokens / 1_000_000.0) * float(rates.get("output") or 0.0)
        result["cost_components_usd"]["output_cost_usd"] = round(output_cost, 8)
        total_cost += output_cost

    result["estimated_cost_usd"] = round(total_cost, 8)
    return result


def unique_clean_terms(items: List[Any], *, limit: int, max_words: int = 8, max_chars: int = 80) -> List[str]:
    out: List[str] = []
    seen = set()
    for raw in items or []:
        term = clean_text(raw)
        term = re.sub(r"^\(?\d+\)?[.)-]?\s*", "", term)
        term = re.sub(r"^[,;:\-]+|[,;:\-]+$", "", term).strip()
        term = term.strip("()[]{}")
        if not term:
            continue
        if re.fullmatch(r"\d+", term):
            continue
        if "(" in term or ")" in term:
            continue
        if len(term) < 2:
            continue
        if len(term) > max_chars:
            continue
        if count_words(term) > max_words:
            continue
        key = normalize_heading_key(term)
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(term)
        if len(out) >= int(limit):
            break
    return out


def normalized_source_text(chapter_title: str, chapter_spec_text: str) -> str:
    return ascii_fold(f"{chapter_title}\n{chapter_spec_text}").lower()


def detect_language_hints(chapter_title: str, chapter_spec_text: str) -> List[str]:
    source = normalized_source_text(chapter_title, chapter_spec_text)
    hints: List[str] = []
    if any(token in source for token in ["entscheidung", "kauf", "wahrgenommen", "unsicher", "leitplanken"]):
        hints.append("de")
    if any(token in source for token in ["decision", "trust", "perceived risk", "uncertainty", "consumer electronics", "choice architecture", "digital nudging"]):
        hints.append("en")
    if not hints:
        hints.append("en" if infer_language_guess(chapter_title + " " + chapter_spec_text) == "en" else "de")
    return list(dict.fromkeys(hints))


def infer_preferred_section_types(chapter_title: str, chapter_spec_text: str) -> List[str]:
    text = normalized_source_text(chapter_title, chapter_spec_text)
    preferred = ["introduction", "background", "discussion", "conclusion", "body_other"]
    if any(token in text for token in ["risk", "trust", "uncertainty", "wahrgenommen", "consumer electronics"]):
        preferred.insert(2, "results")
    if any(token in text for token in ["measurement", "factors", "faktoren", "vergleichbarkeit", "erklarbarkeit", "erklärbarkeit"]):
        preferred.append("methods")
    return list(dict.fromkeys([item for item in preferred if item in PHASE_D_ALLOWED_SECTION_TYPES]))


def infer_penalized_section_types() -> List[str]:
    return ["front_matter", "table_of_contents", "acknowledgements", "references", "appendix", "index"]


def split_chapter_clauses(chapter_spec_text: str) -> List[str]:
    raw = clean_text(chapter_spec_text)
    if not raw:
        return []
    parts = re.split(r"\s*;\s*|\(\d+\)\s*", raw)
    clauses = [clean_text(part) for part in parts if clean_text(part)]
    if clauses:
        return clauses
    return [raw]


def extract_parenthetical_terms(text: str) -> List[str]:
    out: List[str] = []
    for match in re.findall(r"\(([^)]+)\)", str(text or "")):
        for piece in re.split(r"[,/;]", match):
            term = clean_text(piece)
            if term:
                out.append(term)
    return out


def extract_term_candidates(text: str) -> List[str]:
    candidates: List[str] = []
    raw = clean_text(text)
    candidates.extend(extract_parenthetical_terms(raw))
    candidates.extend([clean_text(part) for part in re.split(r"[;,]", raw) if clean_text(part)])
    return unique_clean_terms(candidates, limit=32, max_words=8, max_chars=90)


def build_heuristic_query_plan(chapter_title: str, chapter_spec_text: str, options: PhaseDOptions, stable_hash_fn: Any) -> Dict[str, Any]:
    clauses = split_chapter_clauses(chapter_spec_text)
    title_terms = extract_term_candidates(chapter_title)
    global_terms = extract_term_candidates(chapter_spec_text)
    must_terms = unique_clean_terms(title_terms + global_terms, limit=options.must_term_limit, max_words=6, max_chars=80)
    should_terms = unique_clean_terms(global_terms[1:] + title_terms, limit=options.should_term_limit, max_words=6, max_chars=80)
    preferred_section_types = infer_preferred_section_types(chapter_title, chapter_spec_text)
    language_hints = detect_language_hints(chapter_title, chapter_spec_text)
    subpoints: List[Dict[str, Any]] = []
    for idx, clause in enumerate(clauses[: int(options.subpoint_limit)], start=1):
        clause_terms = extract_term_candidates(clause)
        label = clause_terms[0] if clause_terms else f"Subpoint {idx}"
        subpoints.append(
            {
                "subpoint_id": f"sp_{idx:02d}",
                "label": _truncate(label, max_len=80),
                "summary": clause,
                "must_terms": unique_clean_terms(clause_terms[:6], limit=6, max_words=8, max_chars=90),
                "should_terms": unique_clean_terms(clause_terms[1:10], limit=8, max_words=8, max_chars=90),
                "preferred_section_types": preferred_section_types[:],
            }
        )

    chapter_summary = _truncate(clean_text(chapter_spec_text), max_len=480)
    return {
        "query_id": stable_hash_fn(chapter_title, chapter_spec_text, "phase_d_query_plan", length=16),
        "chapter_title": clean_text(chapter_title),
        "chapter_summary": chapter_summary,
        "must_terms": must_terms,
        "should_terms": should_terms,
        "exclusions": [],
        "subpoints": subpoints,
        "language_hints": language_hints,
        "preferred_section_types": preferred_section_types,
        "penalized_section_types": infer_penalized_section_types(),
        "drift_risks": [],
    }


def build_query_planner_messages(chapter_title: str, chapter_spec_text: str, options: PhaseDOptions) -> List[Dict[str, str]]:
    allowed_types = ", ".join(PHASE_D_ALLOWED_SECTION_TYPES)
    return [
        {
            "role": "system",
            "content": (
                "You create strict query plans for section retrieval over scientific PDFs. "
                "Stay inside the given chapter scope. Do not invent topics. "
                "Return short concrete phrases. Exclusions must be atomic. "
                f"Allowed section types: {allowed_types}."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Chapter title:\n{chapter_title}\n\n"
                f"Chapter spec:\n{chapter_spec_text}\n\n"
                "Build a retrieval query plan for finding useful PDF sections. "
                "The plan should preserve the chapter scope, create several precise subpoints, "
                "and prefer section types that are likely to contain substantive evidence."
            ),
        },
    ]


def term_is_source_anchored(term: str, chapter_title: str, chapter_spec_text: str) -> bool:
    key = normalized_source_text(term, "")
    if not key:
        return False
    source = normalized_source_text(chapter_title, chapter_spec_text)
    if key in source:
        return True
    source_tokens = set(source.split())
    term_tokens = [tok for tok in key.split() if tok]
    if not term_tokens:
        return False
    overlap = sum(1 for tok in term_tokens if tok in source_tokens) / max(1, len(term_tokens))
    return overlap >= 0.75


def source_anchor_terms(items: List[str], chapter_title: str, chapter_spec_text: str, *, limit: int, max_words: int, max_chars: int) -> Dict[str, List[str]]:
    cleaned = unique_clean_terms(items or [], limit=limit, max_words=max_words, max_chars=max_chars)
    kept: List[str] = []
    dropped: List[str] = []
    for item in cleaned:
        if term_is_source_anchored(item, chapter_title, chapter_spec_text):
            kept.append(item)
        else:
            dropped.append(item)
    return {"kept": kept, "dropped": dropped}


def normalize_query_plan(plan_payload: Dict[str, Any], chapter_title: str, chapter_spec_text: str, options: PhaseDOptions, stable_hash_fn: Any) -> Dict[str, Any]:
    baseline_penalized_section_types = infer_penalized_section_types()
    merged_penalized_section_types = list(dict.fromkeys([
        *baseline_penalized_section_types,
        *[item for item in (plan_payload.get("penalized_section_types") or []) if item in PHASE_D_ALLOWED_SECTION_TYPES],
    ]))
    heuristic_plan = build_heuristic_query_plan(chapter_title, chapter_spec_text, options, stable_hash_fn)
    must_term_anchor = source_anchor_terms(plan_payload.get("must_terms") or [], chapter_title, chapter_spec_text, limit=options.must_term_limit, max_words=8, max_chars=90)
    should_term_anchor = source_anchor_terms(plan_payload.get("should_terms") or [], chapter_title, chapter_spec_text, limit=options.should_term_limit, max_words=8, max_chars=90)
    normalized: Dict[str, Any] = {
        "query_id": stable_hash_fn(chapter_title, chapter_spec_text, "phase_d_query_plan", length=16),
        "chapter_title": clean_text(chapter_title),
        "chapter_summary": clean_text(plan_payload.get("chapter_summary") or chapter_spec_text)[:480],
        "must_terms": must_term_anchor["kept"],
        "should_terms": should_term_anchor["kept"],
        "exclusions": unique_clean_terms(plan_payload.get("exclusions") or [], limit=options.exclusion_limit, max_words=6, max_chars=80),
        "subpoints": [],
        "language_hints": unique_clean_terms(plan_payload.get("language_hints") or detect_language_hints(chapter_title, chapter_spec_text), limit=4, max_words=2, max_chars=12),
        "preferred_section_types": [item for item in (plan_payload.get("preferred_section_types") or infer_preferred_section_types(chapter_title, chapter_spec_text)) if item in PHASE_D_ALLOWED_SECTION_TYPES],
        "penalized_section_types": merged_penalized_section_types,
        "drift_risks": unique_clean_terms(plan_payload.get("drift_risks") or [], limit=options.drift_risk_limit, max_words=12, max_chars=120),
        "source_pruned_terms": {
            "must_terms": must_term_anchor["dropped"],
            "should_terms": should_term_anchor["dropped"],
        },
    }

    raw_subpoints = plan_payload.get("subpoints") or []
    for idx, raw in enumerate(raw_subpoints[: int(options.subpoint_limit)], start=1):
        if not isinstance(raw, dict):
            continue
        normalized["subpoints"].append(
            {
                "subpoint_id": clean_text(raw.get("subpoint_id") or f"sp_{idx:02d}")[:32],
                "label": clean_text(raw.get("label") or f"Subpoint {idx}")[:100],
                "summary": clean_text(raw.get("summary") or "")[:320],
                "must_terms": source_anchor_terms(raw.get("must_terms") or [], chapter_title, chapter_spec_text, limit=6, max_words=8, max_chars=90)["kept"],
                "should_terms": source_anchor_terms(raw.get("should_terms") or [], chapter_title, chapter_spec_text, limit=8, max_words=8, max_chars=90)["kept"],
                "preferred_section_types": [item for item in (raw.get("preferred_section_types") or normalized["preferred_section_types"]) if item in PHASE_D_ALLOWED_SECTION_TYPES],
            }
        )
    if not normalized["subpoints"]:
        normalized["subpoints"] = heuristic_plan["subpoints"]

    if not normalized["must_terms"]:
        normalized["must_terms"] = heuristic_plan["must_terms"]
    if not normalized["should_terms"]:
        normalized["should_terms"] = heuristic_plan["should_terms"]
    if not normalized["language_hints"]:
        normalized["language_hints"] = detect_language_hints(chapter_title, chapter_spec_text)
    if not normalized["preferred_section_types"]:
        normalized["preferred_section_types"] = infer_preferred_section_types(chapter_title, chapter_spec_text)
    if not normalized["penalized_section_types"]:
        normalized["penalized_section_types"] = infer_penalized_section_types()

    return normalized


def call_openai_query_planner(chapter_title: str, chapter_spec_text: str, options: PhaseDOptions, stable_hash_fn: Any) -> Dict[str, Any]:
    if OpenAI is None or BaseModel is None:
        raise RuntimeError("OpenAI or pydantic is unavailable for the structured query planner.")
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is missing.")
    client = OpenAI(api_key=OPENAI_API_KEY)
    messages = build_query_planner_messages(chapter_title, chapter_spec_text, options)
    planner_attempts = [
        {
            "model": options.openai_model,
            "reasoning_effort": options.reasoning_effort,
            "max_completion_tokens": options.max_completion_tokens,
            "verbosity": "low",
        },
        {
            "model": options.openai_model,
            "reasoning_effort": "low",
            "max_completion_tokens": max(int(options.max_completion_tokens), 2600),
            "verbosity": "low",
        },
    ]
    if str(options.openai_model or "").strip() != "gpt-5-nano":
        planner_attempts.append(
            {
                "model": "gpt-5-nano",
                "reasoning_effort": "low",
                "max_completion_tokens": max(int(options.max_completion_tokens), 2200),
                "verbosity": "low",
            }
        )

    last_error = None
    attempt_traces: List[Dict[str, Any]] = []
    for attempt in planner_attempts:
        request_kwargs = {
            "model": attempt["model"],
            "messages": messages,
            "response_format": QueryPlanModel,
            "reasoning_effort": attempt["reasoning_effort"],
            "max_completion_tokens": attempt["max_completion_tokens"],
            "verbosity": attempt["verbosity"],
        }
        if not str(attempt["model"] or "").startswith("gpt-5"):
            request_kwargs["temperature"] = options.temperature
        try:
            response = client.beta.chat.completions.parse(**request_kwargs)
            parsed = response.choices[0].message.parsed
            if parsed is None:
                raise RuntimeError("Structured query planner returned no parsed payload.")
            plan_payload = parsed.model_dump()
            normalized_plan = normalize_query_plan(plan_payload, chapter_title, chapter_spec_text, options, stable_hash_fn)
            usage = getattr(response, "usage", None)
            usage_payload = extract_openai_usage_payload(usage)
            cost_payload = estimate_openai_text_cost_usd(str(getattr(response, "model", None) or attempt["model"]), usage_payload)
            return {
                "plan": normalized_plan,
                "planner_trace": {
                    "planner_mode": "openai",
                    "model_requested": options.openai_model,
                    "model_used": str(getattr(response, "model", None) or attempt["model"]),
                    "message_count": len(messages),
                    "attempts": attempt_traces + [attempt],
                    "usage": usage_payload,
                    "cost": cost_payload,
                },
            }
        except Exception as e:
            last_error = e
            attempt_traces.append(
                {
                    "model": attempt["model"],
                    "reasoning_effort": attempt["reasoning_effort"],
                    "max_completion_tokens": attempt["max_completion_tokens"],
                    "verbosity": attempt["verbosity"],
                    "error_type": type(e).__name__,
                    "error_message": str(e),
                }
            )
            continue
    if last_error is not None:
        raise last_error
    raise RuntimeError("Structured query planner failed without returning an error.")


def source_alignment_ratio(plan: Dict[str, Any], chapter_title: str, chapter_spec_text: str) -> float:
    source = normalized_source_text(chapter_title, chapter_spec_text)
    terms = [str(x) for x in (plan.get("must_terms") or []) + (plan.get("should_terms") or []) if str(x).strip()]
    if not terms:
        return 0.0
    hits = 0
    for term in terms:
        key = ascii_fold(term).lower().strip()
        if key and key in source:
            hits += 1
    return round(hits / max(1, len(terms)), 3)


def build_retrieval_views(plan: Dict[str, Any]) -> Dict[str, Any]:
    def join_terms(items: List[str], limit: int = 12) -> str:
        return " | ".join([clean_text(x) for x in (items or []) if clean_text(x)][: int(limit)])

    broad_text = " ".join(
        [
            clean_text(plan.get("chapter_title")),
            clean_text(plan.get("chapter_summary")),
            join_terms(plan.get("should_terms") or [], limit=10),
        ]
    ).strip()
    retrieval_views = {
        "title_lexical": {
            "view_id": "title_lexical",
            "kind": "title_lexical",
            "query_text": " | ".join([clean_text(plan.get("chapter_title")), join_terms(plan.get("must_terms") or [], limit=8)]).strip(" |"),
            "target_units": ["section_title"],
            "preferred_section_types": plan.get("preferred_section_types") or [],
        },
        "summary_semantic": {
            "view_id": "summary_semantic",
            "kind": "summary_semantic",
            "query_text": clean_text(plan.get("chapter_summary")),
            "target_units": ["section_contextualized", "passage_contextualized"],
            "preferred_section_types": plan.get("preferred_section_types") or [],
        },
        "must_terms_lexical": {
            "view_id": "must_terms_lexical",
            "kind": "must_terms_lexical",
            "query_text": join_terms(plan.get("must_terms") or [], limit=12),
            "target_units": ["section_text", "passage_text"],
            "preferred_section_types": plan.get("preferred_section_types") or [],
        },
        "subpoint_views": [],
        "broad_fallback": {
            "view_id": "broad_fallback",
            "kind": "broad_fallback",
            "query_text": broad_text,
            "target_units": ["section_contextualized", "passage_contextualized"],
            "preferred_section_types": plan.get("preferred_section_types") or [],
        },
    }
    for subpoint in plan.get("subpoints") or []:
        retrieval_views["subpoint_views"].append(
            {
                "view_id": f"subpoint::{subpoint.get('subpoint_id')}",
                "kind": "subpoint",
                "label": subpoint.get("label"),
                "query_text": " | ".join(
                    [
                        clean_text(subpoint.get("label")),
                        clean_text(subpoint.get("summary")),
                        join_terms(subpoint.get("must_terms") or [], limit=8),
                    ]
                ).strip(" |"),
                "target_units": ["section_contextualized", "passage_contextualized"],
                "preferred_section_types": subpoint.get("preferred_section_types") or plan.get("preferred_section_types") or [],
            }
        )
    return retrieval_views


def flatten_retrieval_views(retrieval_views: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for key in ["title_lexical", "summary_semantic", "must_terms_lexical", "broad_fallback"]:
        view = retrieval_views.get(key) or {}
        rows.append(
            {
                "view_id": view.get("view_id"),
                "kind": view.get("kind"),
                "target_units": ", ".join(view.get("target_units") or []),
                "preferred_section_types": ", ".join(view.get("preferred_section_types") or []),
                "query_text": _truncate(clean_text(view.get("query_text")), max_len=220),
            }
        )
    for view in retrieval_views.get("subpoint_views") or []:
        rows.append(
            {
                "view_id": view.get("view_id"),
                "kind": view.get("kind"),
                "target_units": ", ".join(view.get("target_units") or []),
                "preferred_section_types": ", ".join(view.get("preferred_section_types") or []),
                "query_text": _truncate(clean_text(view.get("query_text")), max_len=220),
            }
        )
    return rows


def assess_phase_d(plan: Dict[str, Any], retrieval_views: Dict[str, Any], *, options: PhaseDOptions, planner_trace: Dict[str, Any], chapter_title: str, chapter_spec_text: str) -> Dict[str, Any]:
    failures: List[str] = []
    warnings: List[str] = []
    if not clean_text(plan.get("chapter_summary")):
        failures.append("chapter_summary is empty")
    if not (plan.get("must_terms") or []):
        failures.append("must_terms is empty")
    subpoints = plan.get("subpoints") or []
    if len(subpoints) < 1:
        failures.append("subpoints is empty")
    if len(subpoints) > 8:
        warnings.append(f"subpoint count is high ({len(subpoints)})")
    if len(plan.get("drift_risks") or []) < 1:
        warnings.append("drift_risks is empty")
    pruned_terms = (plan.get("source_pruned_terms") or {})
    pruned_count = len(pruned_terms.get("must_terms") or []) + len(pruned_terms.get("should_terms") or [])
    if pruned_count > 0:
        warnings.append(f"planner produced {pruned_count} source-unanchored terms that were pruned")
    if planner_trace.get("planner_mode") != "openai":
        warnings.append("heuristic fallback planner was used")
    alignment = source_alignment_ratio(plan, chapter_title, chapter_spec_text)
    if alignment < 0.35:
        warnings.append(f"source alignment is low ({alignment})")

    view_rows = flatten_retrieval_views(retrieval_views)
    if len(view_rows) < 4:
        failures.append("derived retrieval view count is too low")

    status = "success"
    quality_band = "high"
    if failures:
        status = "failed"
        quality_band = "insufficient"
    elif warnings:
        status = "success_with_warnings"
        quality_band = "acceptable_with_issues"

    qc_rows = [
        qc_row("chapter_summary", "OK" if clean_text(plan.get("chapter_summary")) else "FAIL", bool(clean_text(plan.get("chapter_summary"))), "non-empty", "Retrieval needs a normalized chapter summary.", "inspect query_plan.json and planner trace"),
        qc_row("must_terms", "OK" if (plan.get("must_terms") or []) else "FAIL", len(plan.get("must_terms") or []), ">= 1", "Must terms anchor lexical retrieval.", "inspect planner output or heuristic fallback"),
        qc_row("subpoints", "OK" if 1 <= len(subpoints) <= 8 else "WARN", len(subpoints), "1-8", "Subpoints support diversified retrieval views.", "tighten planner prompt or subpoint limit"),
        qc_row("retrieval_views", "OK" if len(view_rows) >= 4 else "FAIL", len(view_rows), ">= 4", "Phase E should consume multiple derived views.", "inspect retrieval view derivation"),
        qc_row("source_alignment", "OK" if alignment >= 0.35 else "WARN", alignment, ">= 0.35", "Low alignment may indicate topic drift.", "inspect must_terms / should_terms against the chapter text"),
        qc_row("source_pruned_terms", "OK" if pruned_count == 0 else "WARN", pruned_count, "0 preferred", "Pruned planner terms indicate OpenAI drift away from the chapter wording.", "inspect source_pruned_terms in query_plan.json and tighten the planner prompt if needed"),
    ]

    return {
        "status": status,
        "quality_band": quality_band,
        "can_continue_to_next_phase": not failures,
        "failures": failures,
        "warnings": warnings,
        "counts": {
            "must_term_count": len(plan.get("must_terms") or []),
            "should_term_count": len(plan.get("should_terms") or []),
            "subpoint_count": len(subpoints),
            "retrieval_view_count": len(view_rows),
            "source_pruned_term_count": pruned_count,
        },
        "source_alignment_ratio": alignment,
        "qc_rows": qc_rows,
    }


def run_phase_d(run_ctx: Any, *, chapter_title: str, chapter_spec_text: str, options: PhaseDOptions, stable_hash_fn=None, log_event_fn=None, run_logger=None) -> Dict[str, Any]:
    options = options.normalized()
    stable_hash_local = stable_hash_fn or stable_hash
    retrieval_dir = ensure_dir(Path(run_ctx.artifacts.retrieval_dir))
    query_plan_path = Path(run_ctx.artifacts.query_plan_json)
    config_path = retrieval_dir / "phase_d_config.json"
    runtime_path = retrieval_dir / "phase_d_runtime.json"
    query_views_path = retrieval_dir / "query_views.json"
    summary_path = retrieval_dir / "phase_d_summary.json"
    assessment_path = retrieval_dir / "phase_d_assessment.json"
    planner_trace_path = retrieval_dir / "planner_trace.json"

    runtime_payload = {
        "generated_at_utc": utc_now_iso(),
        "phase": "phase_d",
        "options": json_safe(asdict(options)),
        "capabilities": phase_d_capabilities(),
    }
    write_json(runtime_path, runtime_payload)
    write_json(config_path, {"generated_at_utc": utc_now_iso(), "phase": "phase_d", "options": json_safe(asdict(options))})

    if run_logger is not None:
        run_logger.info(
            "Phase D start | openai=%s | api_key=%s | model=%s | allow_heuristic_fallback=%s",
            bool(OpenAI is not None),
            bool(OPENAI_API_KEY),
            options.openai_model,
            options.allow_heuristic_fallback,
        )

    planner_trace: Dict[str, Any]
    if options.use_openai_planner:
        try:
            planner_result = call_openai_query_planner(chapter_title, chapter_spec_text, options, stable_hash_local)
            plan = planner_result["plan"]
            planner_trace = planner_result["planner_trace"]
        except Exception as e:
            if not options.allow_heuristic_fallback:
                raise
            plan = build_heuristic_query_plan(chapter_title, chapter_spec_text, options, stable_hash_local)
            planner_trace = {
                "planner_mode": "heuristic_fallback",
                "planner_error": {"type": type(e).__name__, "message": str(e)},
                "model_requested": options.openai_model,
            }
    else:
        plan = build_heuristic_query_plan(chapter_title, chapter_spec_text, options, stable_hash_local)
        planner_trace = {"planner_mode": "heuristic_only", "model_requested": None}

    retrieval_views = build_retrieval_views(plan)
    assessment = assess_phase_d(plan, retrieval_views, options=options, planner_trace=planner_trace, chapter_title=chapter_title, chapter_spec_text=chapter_spec_text)

    query_plan_payload = {
        "generated_at_utc": utc_now_iso(),
        "phase": "phase_d",
        "query_plan": plan,
        "retrieval_views": retrieval_views,
        "planner_trace": planner_trace,
    }
    write_json(query_plan_path, query_plan_payload)
    write_json(query_views_path, retrieval_views)
    write_json(planner_trace_path, planner_trace)

    view_rows = flatten_retrieval_views(retrieval_views)
    subpoint_rows = [
        {
            "subpoint_id": row.get("subpoint_id"),
            "label": row.get("label"),
            "summary": _truncate(clean_text(row.get("summary")), max_len=180),
            "must_terms": ", ".join(row.get("must_terms") or []),
            "preferred_section_types": ", ".join(row.get("preferred_section_types") or []),
        }
        for row in (plan.get("subpoints") or [])
    ]

    summary_payload = {
        "generated_at_utc": utc_now_iso(),
        "run_id": run_ctx.run_id,
        "phase": "phase_d",
        "options": json_safe(asdict(options)),
        "planner_trace": planner_trace,
        "openai_usage": planner_trace.get("usage"),
        "openai_cost": planner_trace.get("cost"),
        "query_plan_path": rel_to_run(Path(run_ctx.run_dir), query_plan_path),
        "query_views_path": rel_to_run(Path(run_ctx.run_dir), query_views_path),
        "assessment": {k: v for k, v in assessment.items() if k != "qc_rows"},
        "qc_rows": assessment["qc_rows"],
        "subpoints": subpoint_rows,
        "retrieval_views": view_rows,
    }
    assessment_payload = {
        "generated_at_utc": utc_now_iso(),
        "run_id": run_ctx.run_id,
        "phase": "phase_d",
        "assessment": {k: v for k, v in assessment.items() if k != "qc_rows"},
        "qc_rows": assessment["qc_rows"],
        "openai_usage": planner_trace.get("usage"),
        "openai_cost": planner_trace.get("cost"),
        "query_plan_path": rel_to_run(Path(run_ctx.run_dir), query_plan_path),
        "query_views_path": rel_to_run(Path(run_ctx.run_dir), query_views_path),
    }
    write_json(summary_path, summary_payload)
    write_json(assessment_path, assessment_payload)

    if log_event_fn is not None:
        log_event_fn(
            run_ctx,
            stage="phase_d",
            event="phase_finished",
            planner_mode=planner_trace.get("planner_mode"),
            status=assessment["status"],
            subpoint_count=len(subpoint_rows),
            retrieval_view_count=len(view_rows),
            openai_input_tokens=(planner_trace.get("usage") or {}).get("input_tokens"),
            openai_output_tokens=(planner_trace.get("usage") or {}).get("output_tokens"),
            openai_estimated_cost_usd=(planner_trace.get("cost") or {}).get("estimated_cost_usd"),
        )
    if run_logger is not None:
        run_logger.info(
            "Phase D finished | planner_mode=%s | status=%s | subpoints=%s | views=%s | input_tokens=%s | output_tokens=%s | estimated_cost_usd=%s",
            planner_trace.get("planner_mode"),
            assessment["status"],
            len(subpoint_rows),
            len(view_rows),
            (planner_trace.get("usage") or {}).get("input_tokens"),
            (planner_trace.get("usage") or {}).get("output_tokens"),
            (planner_trace.get("cost") or {}).get("estimated_cost_usd"),
        )

    return {
        "config_path": config_path,
        "runtime_path": runtime_path,
        "query_plan_path": query_plan_path,
        "query_views_path": query_views_path,
        "summary_path": summary_path,
        "assessment_path": assessment_path,
        "planner_trace_path": planner_trace_path,
        "planner_trace": planner_trace,
        "query_plan": plan,
        "subpoint_rows": subpoint_rows,
        "retrieval_view_rows": view_rows,
        "assessment": assessment,
        "qc_rows": assessment["qc_rows"],
        "metrics_update": {
            "status": assessment["status"],
            "quality_band": assessment["quality_band"],
            "planner_mode": planner_trace.get("planner_mode"),
            "must_term_count": assessment["counts"].get("must_term_count"),
            "subpoint_count": assessment["counts"].get("subpoint_count"),
            "retrieval_view_count": assessment["counts"].get("retrieval_view_count"),
            "source_alignment_ratio": assessment.get("source_alignment_ratio"),
            "openai_input_tokens": (planner_trace.get("usage") or {}).get("input_tokens"),
            "openai_cached_input_tokens": (planner_trace.get("usage") or {}).get("cached_input_tokens"),
            "openai_output_tokens": (planner_trace.get("usage") or {}).get("output_tokens"),
            "openai_total_tokens": (planner_trace.get("usage") or {}).get("total_tokens"),
            "openai_estimated_cost_usd": (planner_trace.get("cost") or {}).get("estimated_cost_usd"),
            "openai_pricing_model": (planner_trace.get("cost") or {}).get("pricing_model"),
            "openai_pricing_verified_date": (planner_trace.get("cost") or {}).get("pricing_verified_date"),
            "phase_d_summary_path": rel_to_run(Path(run_ctx.run_dir), summary_path),
            "phase_d_assessment_path": rel_to_run(Path(run_ctx.run_dir), assessment_path),
        },
    }


In [ ]:
# Phase D.1 - Build the query plan and derived retrieval views

phase_d_options = PhaseDOptions(
    force_rebuild=True,
    use_openai_planner=True,
    allow_heuristic_fallback=True,
    openai_model=(os.getenv("OPENAI_PDF_SCAN_PLANNER_MODEL") or os.getenv("OPENAI_PDF_SCAN_MODEL") or "gpt-5-mini").strip() or "gpt-5-mini",
    reasoning_effort="medium",
    temperature=0.0,
    max_completion_tokens=1800,
    must_term_limit=12,
    should_term_limit=18,
    exclusion_limit=8,
    subpoint_limit=6,
    drift_risk_limit=8,
)

phase_d_logger = setup_run_logger(RUN_CONTEXT)

with stage_timer(RUN_CONTEXT, "phase_d"):
    phase_d_result = run_phase_d(
        RUN_CONTEXT,
        chapter_title=CHAPTER_TITLE,
        chapter_spec_text=CHAPTER_DESCRIPTION,
        options=phase_d_options,
        stable_hash_fn=stable_hash,
        log_event_fn=log_event,
        run_logger=phase_d_logger,
    )
    metrics = load_metrics(RUN_CONTEXT)
    metrics.setdefault("stages", {}).setdefault("phase_d", {}).update(phase_d_result["metrics_update"])
    save_metrics(RUN_CONTEXT, metrics)

phase_d_rel = lambda path: rel_to_run(Path(RUN_CONTEXT.run_dir), Path(path))

PHASE_D_RESULT = phase_d_result
PHASE_D_PLAN = phase_d_result["query_plan"]
PHASE_D_SUBPOINTS = phase_d_result["subpoint_rows"]
PHASE_D_VIEWS = phase_d_result["retrieval_view_rows"]

print_section("Phase D - Planner Capabilities")
print_kv(
    {
        "openai_available": phase_d_capabilities().get("openai_available"),
        "pydantic_available": phase_d_capabilities().get("pydantic_available"),
        "openai_api_key_present": phase_d_capabilities().get("openai_api_key_present"),
        "planner_model": phase_d_options.openai_model,
        "planner_mode": phase_d_result["planner_trace"].get("planner_mode"),
        "pricing_source_url": (phase_d_result["planner_trace"].get("cost") or {}).get("pricing_source_url"),
        "pricing_verified_date": (phase_d_result["planner_trace"].get("cost") or {}).get("pricing_verified_date"),
        "source_alignment_ratio": phase_d_result["assessment"].get("source_alignment_ratio"),
    }
)

print_section("Phase D - What Happened")
print_kv(
    {
        "query_plan_json": phase_d_rel(phase_d_result["query_plan_path"]),
        "query_views_json": phase_d_rel(phase_d_result["query_views_path"]),
        "phase_d_config_json": phase_d_rel(phase_d_result["config_path"]),
        "phase_d_runtime_json": phase_d_rel(phase_d_result["runtime_path"]),
        "phase_d_summary_json": phase_d_rel(phase_d_result["summary_path"]),
        "phase_d_assessment_json": phase_d_rel(phase_d_result["assessment_path"]),
        "planner_trace_json": phase_d_rel(phase_d_result["planner_trace_path"]),
        "must_terms": len(PHASE_D_PLAN.get("must_terms") or []),
        "subpoints": len(PHASE_D_SUBPOINTS),
        "retrieval_views": len(PHASE_D_VIEWS),
        "openai_input_tokens": (phase_d_result["planner_trace"].get("usage") or {}).get("input_tokens"),
        "openai_cached_input_tokens": (phase_d_result["planner_trace"].get("usage") or {}).get("cached_input_tokens"),
        "openai_output_tokens": (phase_d_result["planner_trace"].get("usage") or {}).get("output_tokens"),
        "openai_total_tokens": (phase_d_result["planner_trace"].get("usage") or {}).get("total_tokens"),
        "openai_estimated_cost_usd": (phase_d_result["planner_trace"].get("cost") or {}).get("estimated_cost_usd"),
        "phase_status": phase_d_result["assessment"].get("status"),
    }
)

print_section("Phase D - Query Plan Summary")
print_kv(
    {
        "chapter_title": _truncate(PHASE_D_PLAN.get("chapter_title"), max_len=110),
        "chapter_summary": _truncate(PHASE_D_PLAN.get("chapter_summary"), max_len=220),
        "must_terms": ", ".join(PHASE_D_PLAN.get("must_terms") or []),
        "should_terms": ", ".join((PHASE_D_PLAN.get("should_terms") or [])[:12]),
        "exclusions": ", ".join(PHASE_D_PLAN.get("exclusions") or []) or "<none>",
        "language_hints": ", ".join(PHASE_D_PLAN.get("language_hints") or []),
        "preferred_section_types": ", ".join(PHASE_D_PLAN.get("preferred_section_types") or []),
        "penalized_section_types": ", ".join(PHASE_D_PLAN.get("penalized_section_types") or []),
        "drift_risks": ", ".join(PHASE_D_PLAN.get("drift_risks") or []) or "<none>",
    }
)

print_section("Phase D - Subpoints")
print_table(
    PHASE_D_SUBPOINTS,
    columns=["subpoint_id", "label", "summary", "must_terms", "preferred_section_types"],
    max_rows=20,
    max_col_width=48,
)

print_section("Phase D - Retrieval Views")
print_table(
    PHASE_D_VIEWS,
    columns=["view_id", "kind", "target_units", "preferred_section_types", "query_text"],
    max_rows=30,
    max_col_width=52,
)

print_section("Phase D - Assessment")
print_kv(
    {
        "status": phase_d_result["assessment"].get("status"),
        "quality_band": phase_d_result["assessment"].get("quality_band"),
        "can_continue": phase_d_result["assessment"].get("can_continue_to_next_phase"),
        "warning_count": len(phase_d_result["assessment"].get("warnings") or []),
        "failure_count": len(phase_d_result["assessment"].get("failures") or []),
        "planner_mode": phase_d_result["planner_trace"].get("planner_mode"),
        "estimated_cost_usd": (phase_d_result["planner_trace"].get("cost") or {}).get("estimated_cost_usd"),
    }
)

print_section("Phase D - QC")
print_table(
    phase_d_result["qc_rows"],
    columns=["check", "status", "value", "expected", "why", "fix"],
    max_rows=20,
    max_col_width=46,
)
